## Cell 1 — Imports, configuration, and global setup

This cell initializes the notebook environment for the LSTM experiment. It imports all required Python libraries and project-specific helper modules, loads the experiment configuration from the YAML file, and extracts the most important parameters such as the train/validation split, LSTM hyperparameters, trading threshold, and selected ticker.

In addition, the cell creates the main output folders used throughout the pipeline, including the experiment-specific run directory where models, logs, plots, and result files can later be stored. This provides a single, consistent setup point for the rest of the notebook.


In [ ]:
# ============================================================
# Cell 1 — Imports, configuration, and global setup
# Purpose:
# - import required libraries and project modules
# - reload local project modules so notebook runs reflect file edits
# - load experiment configuration from YAML
# - define global parameters used across the notebook
# - create output folders for results and saved artifacts
# ============================================================

import os
import importlib
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

import data_loader
import indicators
import preprocessing
import model
import evaluation

importlib.reload(data_loader)
importlib.reload(indicators)
importlib.reload(preprocessing)
importlib.reload(model)
importlib.reload(evaluation)

load_stock_data = data_loader.load_stock_data
add_indicators = indicators.add_indicators
impute_numeric_median = preprocessing.impute_numeric_median
df_to_windowed_df = preprocessing.df_to_windowed_df
windowed_df_to_date_X_y = preprocessing.windowed_df_to_date_X_y
build_lstm_model = model.build_lstm_model
compute_metrics = evaluation.compute_metrics
simulate_pnl = evaluation.simulate_pnl
compute_signal_accuracy_from_reference = evaluation.compute_signal_accuracy_from_reference
compute_directional_accuracy_from_reference = evaluation.compute_directional_accuracy_from_reference

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------
CONFIG_PATH = Path("config/config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# ------------------------------------------------------------
# Read split parameters
# ------------------------------------------------------------
sp = config.get("split", {})
TRAIN_END = float(sp.get("train_end", 0.70))
VAL_END = float(sp.get("val_end", 0.80))

# ------------------------------------------------------------
# Read LSTM hyperparameters
# ------------------------------------------------------------
lstm_cfg = config["lstm"]
window_size = int(lstm_cfg["window_size"])
epochs = int(lstm_cfg["epochs"])
batch_size = int(lstm_cfg["batch_size"])
learning_rate = float(lstm_cfg["learning_rate"])

# ------------------------------------------------------------
# Trading / feature settings
# ------------------------------------------------------------
TRADING_THR = float(config.get("trading", {}).get("threshold_pct", 0.0))
USE_STATIONARY_FEATURES = bool(config.get("features", {}).get("stationary", False))

target_cfg = config.get("target", {}) or {}
TARGET_TYPE = str(target_cfg.get("type", "price")).strip().lower()
RETURN_TYPE = str(target_cfg.get("return_type", "simple")).strip().lower()
DIRECTION_CUTOFF = float(target_cfg.get("direction_cutoff", 0.0))
DIRECTION_PROB_THRESHOLD = float(target_cfg.get("direction_probability_threshold", 0.5))
TASK_TYPE = "classification" if TARGET_TYPE == "direction" else "regression"

if TARGET_TYPE not in {"price", "return", "direction"}:
    raise ValueError("target.type must be one of: price, return, direction")

if RETURN_TYPE not in {"simple", "log"}:
    raise ValueError("target.return_type must be one of: simple, log")

wf_cfg = config.get("walk_forward", {}) or {}
USE_WALK_FORWARD = bool(wf_cfg.get("enable", False))
WF_EXPANDING = bool(wf_cfg.get("expanding", True))
WF_MIN_TRAIN_SIZE = wf_cfg.get("min_train_size", 0.50)
WF_VAL_SIZE = wf_cfg.get("val_size", 0.10)
WF_STEP_SIZE = wf_cfg.get("step_size", WF_VAL_SIZE)

# ------------------------------------------------------------
# Paths and ticker
# ------------------------------------------------------------
ticker = config["tickers"][0]

csv_folder = Path(config["paths"]["csv_folder"])
model_dir = Path(config["paths"]["model_folder_lstm"])
output_dir = Path("output")
runs_dir = Path("runs")
run_root = runs_dir / f"{ticker}_{TARGET_TYPE}_experiment"

output_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
run_root.mkdir(parents=True, exist_ok=True)

if TARGET_TYPE == "direction":
    SEARCH_SORT_COLS = ["val_f1", "val_acc", "val_auc", "val_pnl_ratio"]
    SEARCH_SORT_ASC = [False, False, False, False]
else:
    SEARCH_SORT_COLS = ["val_rmse", "val_r2", "val_da", "val_sa", "val_pnl_ratio"]
    SEARCH_SORT_ASC = [True, False, False, False, False]


def sort_results_for_selection(df_):
    return df_.sort_values(SEARCH_SORT_COLS, ascending=SEARCH_SORT_ASC)


print("Target type:", TARGET_TYPE)
print("Task type:", TASK_TYPE)
print("Walk-forward enabled:", USE_WALK_FORWARD)
print("Run folder:", run_root)
print("Walk-forward expanding:", WF_EXPANDING)
print("Walk-forward min train size:", WF_MIN_TRAIN_SIZE)
print("Walk-forward validation size:", WF_VAL_SIZE)
print("Walk-forward step size:", WF_STEP_SIZE)
print("Ticker:", ticker)
print("Split (train_end, val_end):", TRAIN_END, VAL_END)
print("Window size:", window_size)
print("Epochs:", epochs)
print("Batch size:", batch_size)
print("Learning rate:", learning_rate)
print("Trading threshold:", TRADING_THR)
print("Run folder:", run_root)

## Cell 2 — Load and prepare raw price data

This cell loads the raw stock-price dataset for the selected ticker and prepares it for the later feature-engineering steps.

The code first tries to read a local CSV file from the configured data folder. If no CSV file is found, it falls back to the project’s `load_stock_data()` function. After loading, the `Date` column is converted into a timezone-naive datetime format to ensure consistency across different data sources.

Only the core market columns required for the experiment are retained, namely `Date`, `Open`, `High`, `Low`, `Close`, and `Volume` if available. In addition, an internal `__DATE__` column is created and used as the standard chronological reference throughout the pipeline. Finally, the dataset is sorted by date and a short preview is displayed as a sanity check.

In [ ]:
# ============================================================
# Cell 2 — Load and prepare raw price data
# Purpose:
# - load the selected ticker dataset from CSV or fallback loader
# - standardize the Date column
# - keep only the required raw market columns
# - create a consistent internal __DATE__ column
# - sort data chronologically for later time-series processing
# ============================================================

# ------------------------------------------------------------
# Load raw price data for the selected ticker
# Priority:
# 1) local CSV file
# 2) fallback to project loader
# ------------------------------------------------------------
csv_path = csv_folder / f"{ticker}.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)

    if "Date" not in df.columns:
        raise ValueError(
            f"'Date' column not found in {csv_path}. Available columns: {list(df.columns)}"
        )

    # Convert Date to timezone-naive datetime
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)

else:
    df_list = load_stock_data(str(csv_folder), [ticker])

    if not df_list or len(df_list) == 0:
        raise ValueError(f"No data returned for ticker '{ticker}'.")

    df = df_list[0].reset_index()

    if "Date" not in df.columns:
        raise ValueError(
            f"'Date' column missing after load_stock_data() for ticker '{ticker}'."
        )

    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)


# ------------------------------------------------------------
# Keep only the raw columns needed downstream
# ------------------------------------------------------------
raw_cols = [c for c in ["Date", "Open", "High", "Low", "Close", "Volume"] if c in df.columns]
df = df[raw_cols].copy()


# ------------------------------------------------------------
# Basic validation of required columns
# ------------------------------------------------------------
required_cols = ["Date", "Close"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


# ------------------------------------------------------------
# Create a consistent internal date column
# This avoids later issues with index/column differences
# ------------------------------------------------------------
df["__DATE__"] = pd.to_datetime(df["Date"])


# ------------------------------------------------------------
# Sort chronologically and reset index
# ------------------------------------------------------------
df = df.sort_values("__DATE__").reset_index(drop=True)


# ------------------------------------------------------------
# Quick sanity checks
# ------------------------------------------------------------
print("✅ Raw data loaded:", df.shape)
print("Date range:", df["__DATE__"].min(), "->", df["__DATE__"].max())
print("Columns:", list(df.columns))

display(df.head(3))
display(df.tail(3))

## Cell 3 — Add technical indicators

This cell performs the feature-engineering step of the pipeline by adding the configured technical indicators to the raw stock-price dataset.

Using the add_indicators() function together with the settings from config.yaml, the code computes the active technical indicators such as RSI, MACD, moving averages, Bollinger Bands, Momentum, and Rolling Volatility. Log-return features and directional regime features are only created if they are explicitly enabled in the configuration.

Because many technical indicators rely on rolling windows, the first observations of the dataset usually contain missing values. Therefore, the cell counts the number of rows affected by NaN values, removes them, and reports how many observations were lost due to indicator construction.

Finally, the code verifies that the internal __DATE__ column is still available, sorts the dataset chronologically, and prints a preview of the final feature-engineered DataFrame. This cleaned dataset, stored as df_ind, is then used in the next step to define the prediction target.

In [ ]:
# ============================================================
# Cell 3 — Add technical indicators
# Purpose:
# - create technical indicator features from raw OHLCV data
# - inspect NaNs created by rolling calculations
# - validate date consistency
# - sort chronologically
# - show a preview of the engineered dataset
# ============================================================

# ------------------------------------------------------------
# Apply feature engineering
# This adds the technical indicators defined in indicators.py
# using the settings from config["technical_indicators"].
# ------------------------------------------------------------
df_ind = add_indicators(
    df.copy(),
    config.get("technical_indicators", {})
).copy()


# ------------------------------------------------------------
# Check how many rows contain NaN values after indicator creation
# Rolling indicators such as RSI, SMA, EMA, Bollinger Bands,
# MACD, and volatility naturally create NaNs at the beginning.
# ------------------------------------------------------------
rows_before = len(df_ind)
nan_rows = df_ind.isna().any(axis=1).sum()

print(f"Rows before NaN handling: {rows_before}")
print(f"Rows containing NaNs after indicators: {nan_rows}")


# ------------------------------------------------------------
# Remove rows with NaNs created by rolling indicator windows
# This keeps the modeling dataset clean and avoids unstable
# behavior in later scaling and windowing steps.
# ------------------------------------------------------------
df_ind = df_ind.dropna().reset_index(drop=True)

rows_after = len(df_ind)
print(f"Rows after dropping NaNs: {rows_after}")
print(f"Rows removed due to indicator windows: {rows_before - rows_after}")


# ------------------------------------------------------------
# Ensure the internal date column still exists after feature creation
# ------------------------------------------------------------
if "__DATE__" not in df_ind.columns:
    if "Date" in df_ind.columns:
        df_ind["__DATE__"] = pd.to_datetime(df_ind["Date"])
    elif isinstance(df_ind.index, pd.DatetimeIndex):
        df_ind["__DATE__"] = pd.to_datetime(df_ind.index)
    else:
        raise ValueError(
            "After feature engineering, no '__DATE__' or 'Date' column was found."
        )


# ------------------------------------------------------------
# Sort chronologically for safe downstream time-series processing
# ------------------------------------------------------------
df_ind = df_ind.sort_values("__DATE__").reset_index(drop=True)


# ------------------------------------------------------------
# Show engineered dataset preview
# ------------------------------------------------------------
print("✅ Indicator dataset ready:", df_ind.shape)

feature_cols = [c for c in df_ind.columns if c not in ["Date", "__DATE__"]]
print("Engineered feature columns:")
print(feature_cols)

display(df_ind.head(3))
display(df_ind.tail(3))

## Cell 4 — Create the prediction target and final modeling dataset

This cell prepares the final dataset used for model training.

The notebook supports three target modes controlled by the configuration: next-day closing price, next-day return, and next-day direction. All helper target columns are created first, after which the active modeling target is selected according to target.type in config.yaml.

Since the final observation has no future value available, its target becomes missing and is removed. Any infinite values introduced during feature engineering are replaced with missing values so they can be handled safely downstream.

The resulting DataFrame, stored as df_work, contains the cleaned feature set together with the active target variable and serves as the main input for the later scaling, windowing, and model-training steps.

In [ ]:
# ============================================================
# Cell 4A — Create target and final modeling dataset
# Purpose:
# - define the prediction target
# - clean invalid values
# - build the final dataset used for modeling
# ============================================================

# ------------------------------------------------------------
# Define the prediction target:
# Target = next-day closing price (T+1)
# ------------------------------------------------------------
df_ind["Target"] = df_ind["Close"].shift(-1)

# If later you want price-difference prediction instead,
# you would use:
# df_ind["Target"] = df_ind["Close"].shift(-1) - df_ind["Close"]

# ------------------------------------------------------------
# Remove the last row because it has no future target value
# ------------------------------------------------------------
df_work = df_ind.dropna(subset=["Target"]).copy()

# ------------------------------------------------------------
# Replace infinite values that may occur in engineered features
# ------------------------------------------------------------
df_work = df_work.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# Final sanity check
# ------------------------------------------------------------
df_ind["Target_NextPrice"] = df_ind["Close"].shift(-1)

if RETURN_TYPE == "log":
    df_ind["Target_Return"] = np.log(df_ind["Target_NextPrice"] / df_ind["Close"])
else:
    df_ind["Target_Return"] = (df_ind["Target_NextPrice"] / df_ind["Close"]) - 1.0

df_ind["Target_Direction"] = np.where(
    df_ind["Target_Return"].notna(),
    (df_ind["Target_Return"] > DIRECTION_CUTOFF).astype(int),
    np.nan
)

TARGET_SOURCE_COL = {
    "price": "Target_NextPrice",
    "return": "Target_Return",
    "direction": "Target_Direction",
}[TARGET_TYPE]

df_ind["Target"] = df_ind[TARGET_SOURCE_COL]

df_work = df_ind.dropna(subset=["Target_NextPrice", "Target_Return", "Target"]).copy()
df_work = df_work.replace([np.inf, -np.inf], np.nan)

if TARGET_TYPE == "direction":
    df_work["Target"] = df_work["Target"].astype(int)
    df_work["Target_Direction"] = df_work["Target_Direction"].astype(int)

print("✅ Final modeling dataset ready:", df_work.shape)
print("Target mode:", TARGET_TYPE)
print("Target source column:", TARGET_SOURCE_COL)
display(df_work.head(3))
display(df_work.tail(3))

## Cell 5 — Stationarity testing of input features

Before training the LSTM model, this cell evaluates whether the numeric input features are stationary.

In financial time-series analysis, stationarity refers to the property that the statistical behavior of a variable—such as its mean, variance, and autocorrelation structure—remains stable over time. Since many raw price-based variables are non-stationary, it is useful to document this property before model training.

Two complementary statistical tests are applied:

- **Augmented Dickey–Fuller (ADF) test**  
  Tests the null hypothesis that the series is non-stationary due to a unit root.  
  A p-value below 0.05 suggests stationarity.

- **KPSS test**  
  Tests the null hypothesis that the series is stationary.  
  A p-value above 0.05 suggests stationarity.

Based on the combination of both tests, each feature is classified as:

- **stationary**
- **non_stationary**
- **inconclusive**

The results are saved as a CSV file so they can be referenced later in the thesis when discussing preprocessing choices and feature behavior.

In [ ]:
# ============================================================
# Cell 5 — Stationarity testing of input features
# Purpose:
# - test numeric / boolean input features on the INITIAL TRAIN block only
# - classify them using ADF and KPSS
# - optionally activate stationary-only feature filtering
# ============================================================

from statsmodels.tsa.stattools import adfuller, kpss

# ------------------------------------------------------------
# Run stationarity tests for one series
# ------------------------------------------------------------
def _stationarity_test_one(series: pd.Series):
    s = pd.to_numeric(series, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan).dropna()

    if len(s) < 30:
        return np.nan, np.nan, "inconclusive", "too_short"

    if float(s.nunique()) <= 1:
        return np.nan, np.nan, "inconclusive", "constant"

    try:
        adf_p = float(adfuller(s, autolag="AIC")[1])
    except Exception:
        return np.nan, np.nan, "inconclusive", "adf_fail"

    try:
        kpss_p = float(kpss(s, regression="c", nlags="auto")[1])
    except Exception:
        return adf_p, np.nan, "inconclusive", "kpss_fail"

    if (adf_p < 0.05) and (kpss_p > 0.05):
        verdict = "stationary"
    elif (adf_p > 0.05) and (kpss_p < 0.05):
        verdict = "non_stationary"
    else:
        verdict = "inconclusive"

    return adf_p, kpss_p, verdict, "ok"


# ------------------------------------------------------------
# Run stationarity report for a whole DataFrame
# ------------------------------------------------------------
def stationarity_report(df: pd.DataFrame, feature_cols=None, exclude_cols=None, save_path=None):
    if feature_cols is None:
        feature_cols = list(df.columns)

    default_exclude = {
        "__DATE__", "Date", "date", "Datetime", "datetime",
        "timestamp", "Target", "Target_NextPrice", "Target_Return", "Target_Direction"
    }

    exclude_cols = set(exclude_cols or set()) | default_exclude
    rows = []

    for col in feature_cols:
        if col in exclude_cols:
            continue
        if col not in df.columns:
            continue

        is_numeric_like = (
            pd.api.types.is_numeric_dtype(df[col]) or
            pd.api.types.is_bool_dtype(df[col])
        )

        if not is_numeric_like:
            rows.append({
                "feature": col,
                "ADF_p": np.nan,
                "KPSS_p": np.nan,
                "verdict": "inconclusive",
                "status": "non_numeric"
            })
            continue

        adf_p, kpss_p, verdict, status = _stationarity_test_one(df[col])

        rows.append({
            "feature": col,
            "ADF_p": adf_p,
            "KPSS_p": kpss_p,
            "verdict": verdict,
            "status": status
        })

    rep = pd.DataFrame(rows)

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        rep.to_csv(save_path, index=False)

    return rep


# ------------------------------------------------------------
# IMPORTANT:
# Compute stationarity on the INITIAL TRAIN BLOCK ONLY.
# This avoids using validation/test data for feature filtering.
# ------------------------------------------------------------
stationarity_path = f"output/{ticker}_stationarity_report.csv"

stationarity_cutoff = max(1, min(int(len(df_work) * TRAIN_END), len(df_work) - 1))
df_stationarity = df_work.iloc[:stationarity_cutoff].copy()

rep = stationarity_report(
    df_stationarity,
    save_path=stationarity_path
)

# ------------------------------------------------------------
# Build the active feature universe
# ------------------------------------------------------------
NON_FEATURE_COLS = {
    "__DATE__", "Date", "date", "Datetime", "datetime",
    "timestamp", "Target", "Target_NextPrice", "Target_Return", "Target_Direction"
}

ALL_CANDIDATE_FEATURES = [
    c for c in df_work.columns
    if (c not in NON_FEATURE_COLS)
    and (
        pd.api.types.is_numeric_dtype(df_work[c]) or
        pd.api.types.is_bool_dtype(df_work[c])
    )
]

STATIONARY_FEATURES = set(
    rep.loc[rep["verdict"] == "stationary", "feature"].astype(str).tolist()
)

if USE_STATIONARY_FEATURES:
    ACTIVE_FEATURE_SET = set(
        c for c in ALL_CANDIDATE_FEATURES
        if c in STATIONARY_FEATURES
    )
else:
    ACTIVE_FEATURE_SET = set(ALL_CANDIDATE_FEATURES)

if USE_STATIONARY_FEATURES and len(ACTIVE_FEATURE_SET) == 0:
    raise ValueError(
        "USE_STATIONARY_FEATURES=True, but no stationary features remained "
        "after train-only stationarity filtering."
    )

# Always keep Close available as evaluation reference,
# even if it is not used as a model feature.
REFERENCE_PRICE_COL = "Close"
if REFERENCE_PRICE_COL not in df_work.columns:
    raise ValueError("'Close' must exist in df_work for directional evaluation.")

print("✅ Stationarity report created:", rep.shape)
print("Saved to:", stationarity_path)
print("Stationarity sample length (train-only):", len(df_stationarity))
print("USE_STATIONARY_FEATURES:", USE_STATIONARY_FEATURES)
print("Active feature count:", len(ACTIVE_FEATURE_SET))
display(rep.head(15))
print("Active features:")
print(sorted(ACTIVE_FEATURE_SET))

### Cell 6 — Define feature groups for brute-force experiments

This cell defines the feature structure used in the LSTM experiments.

A base set of market variables, such as Close, Open, High, Low, and Volume, is included whenever available. In addition, the active technical indicators are organized into logical groups such as RSI, MACD, moving averages, Bollinger Bands, Momentum, and Volatility. Only groups that exist in the engineered dataset and pass the optional stationary-feature filter remain active.

Grouping the indicators in this way makes it possible to systematically test all non-empty feature combinations in the brute-force experiment. Before training begins, the code also checks whether each feature is actually present in the dataset and removes empty groups automatically.

In [ ]:
# ============================================================
# Cell 6 — Feature group definitions
# Purpose:
# - define base input features
# - define indicator groups for brute-force experiments
# - apply availability filtering
# - apply optional stationary-only filtering
# ============================================================

def apply_active_feature_filter(cols):
    cols = [c for c in cols if c in df_work.columns]
    cols = [c for c in cols if c in ACTIVE_FEATURE_SET]
    return cols


# Base market features
BASE_FEATURES = apply_active_feature_filter(["Close", "Open", "High", "Low", "Volume"])

# Technical indicator groups
INDICATOR_GROUPS = {
    "RSI": ["RSI"],
    "MACD": ["MACD", "MACD_signal", "MACD_hist"],
    "SMA": ["SMA"],
    "EMA": ["EMA"],
    "BB": ["BB_upper", "BB_lower", "BB_middle"],
    "MOM": ["Momentum"],
    "VOL": ["RollingVolatility"],

}

# Apply availability + optional stationarity filter
INDICATOR_GROUPS = {
    g: apply_active_feature_filter(cols)
    for g, cols in INDICATOR_GROUPS.items()
}

# Remove empty groups
INDICATOR_GROUPS = {
    g: cols for g, cols in INDICATOR_GROUPS.items()
    if len(cols) > 0
}


def parse_combo_name(combo_name: str):
    """
    Convert a stored combination string into a list of indicator groups.

    Examples:
        'RSI+MACD+LOGRET' -> ['RSI', 'MACD', 'LOGRET']
        'BASEONLY' -> []
    """
    if (combo_name is None) or (str(combo_name).strip().upper() == "BASEONLY") or (str(combo_name).strip() == ""):
        return []
    return str(combo_name).split("+")


def combo_has_group(combo_name: str, group_name: str) -> bool:
    return group_name in parse_combo_name(combo_name)


def groups_to_feature_cols(groups):
    """
    Convert a list of indicator-group names into the actual feature columns
    used by the model.
    """
    cols = []
    for g in groups:
        cols += INDICATOR_GROUPS.get(g, [])
    feature_cols = list(dict.fromkeys(BASE_FEATURES + cols))
    return feature_cols


print("✅ BASE_FEATURES:", BASE_FEATURES)
print("✅ INDICATOR_GROUPS:")
for k, v in INDICATOR_GROUPS.items():
    print(" ", k, "->", v)

n_groups = len(INDICATOR_GROUPS)
n_indicator_combinations = (2 ** n_groups) - 1
n_total_model_combinations = n_indicator_combinations + (1 if len(BASE_FEATURES) > 0 else 0)

print(f"\nTotal indicator groups: {n_groups}")
print(f"Indicator-group non-empty combinations: {n_indicator_combinations}")
print(f"Total model combinations including BASEONLY: {n_total_model_combinations}")

if len(BASE_FEATURES) == 0:
    print("⚠️ BASEONLY will be skipped because no base features remain after filtering.")

if (len(BASE_FEATURES) == 0) and (n_groups == 0):
    raise ValueError(
        "No usable model features remain after availability/stationarity filtering."
    )

### Cell 7 — Task-aware preprocessing and validation helpers

This cell defines the helper functions used to prepare model inputs, normalize data without leakage, build windows, train models, and evaluate them under the active task configuration.

The helpers support all three target modes used in the notebook: price regression, return regression, and direction classification. Feature scalers are always fitted on the training split only and then reused for validation and test data. For direction classification, the target is not scaled.

The same cell also contains the walk-forward validation helpers used when walk_forward.enable is set to true. This keeps the search logic and the final holdout evaluation consistent inside a single task-aware implementation.

In [ ]:
# ============================================================
# Cell 7 — Task-aware preprocessing, modeling, and validation helpers
# Purpose:
# - build leakage-free train/validation/test helpers
# - support price, return, and direction targets
# - support both single-split and walk-forward validation
# ============================================================

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

REFERENCE_COLS = ["Close", "Target_NextPrice", "Target_Return", "Target_Direction"]


def _to_daily_index(values):
    idx = pd.DatetimeIndex(pd.to_datetime(values, errors="coerce"))
    if idx.isna().any():
        raise ValueError("Invalid datetime values found during date alignment.")
    return idx.normalize()


def make_numeric_model_df(df_work, feature_cols):
    keep = [c for c in feature_cols if c in df_work.columns]
    if len(keep) == 0:
        raise ValueError("No feature columns available for the current combination.")

    df_in = df_work[keep + ["Target"]].copy()

    bool_cols = [c for c in df_in.columns if pd.api.types.is_bool_dtype(df_in[c])]
    if bool_cols:
        df_in[bool_cols] = df_in[bool_cols].astype(np.int8)

    df_in = df_in.select_dtypes(include=[np.number]).copy()
    df_in = df_in.replace([np.inf, -np.inf], np.nan)
    df_in = df_in.dropna(subset=["Target"]).copy()

    if df_in.shape[1] <= 1:
        raise ValueError("No numeric feature columns remain after preprocessing.")

    return df_in


def attach_datetime_index(df_numeric, df_source):
    row_idx = df_numeric.index.copy()

    if "__DATE__" in df_source.columns:
        date_index = _to_daily_index(df_source.loc[row_idx, "__DATE__"])
    elif "Date" in df_source.columns:
        date_index = _to_daily_index(df_source.loc[row_idx, "Date"])
    else:
        raise ValueError("No date column found (expected '__DATE__' or 'Date').")

    df_out = df_numeric.copy()
    df_out.index = date_index
    return df_out.sort_index()


def build_reference_frame(df_source, row_idx, ref_cols=None):
    ref_cols = [c for c in (ref_cols or REFERENCE_COLS) if c in df_source.columns]
    missing_required = [c for c in ["Close", "Target_NextPrice"] if c not in ref_cols]
    if missing_required:
        raise ValueError(f"Missing required reference columns: {missing_required}")

    ref = df_source.loc[row_idx, ref_cols].copy()

    if "__DATE__" in df_source.columns:
        ref.index = _to_daily_index(df_source.loc[row_idx, "__DATE__"])
    elif "Date" in df_source.columns:
        ref.index = _to_daily_index(df_source.loc[row_idx, "Date"])
    else:
        raise ValueError("No date column found (expected '__DATE__' or 'Date').")

    return ref.sort_index()


def reference_frame_for_dates(ref_df, dates):
    ref_out = ref_df.copy()
    ref_out.index = _to_daily_index(ref_out.index)
    target_dates = _to_daily_index(np.asarray(dates).reshape(-1))
    out = ref_out.reindex(target_dates)

    if out.isna().any().any():
        raise ValueError("Missing reference rows after date alignment.")

    return out


def fit_train_only_preprocessor(df_train, target_col="Target", task_type=TASK_TYPE):
    df_train = df_train.copy()
    feature_cols = [c for c in df_train.columns if c != target_col]
    if len(feature_cols) == 0:
        raise ValueError("No feature columns available for scaling.")

    train_feature_medians = df_train[feature_cols].median()
    train_target_median = df_train[target_col].median()

    df_train[feature_cols] = df_train[feature_cols].fillna(train_feature_medians)
    df_train[target_col] = df_train[target_col].fillna(train_target_median)

    scaler_X = MinMaxScaler().fit(df_train[feature_cols])
    scaler_y = None if task_type == "classification" else MinMaxScaler().fit(df_train[[target_col]])

    return {
        "task_type": task_type,
        "feature_cols": feature_cols,
        "train_feature_medians": train_feature_medians,
        "train_target_median": train_target_median,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
    }


def transform_with_preprocessor(df_any, prep, target_col="Target"):
    df_any = df_any.copy()
    feature_cols = prep["feature_cols"]

    df_any[feature_cols] = df_any[feature_cols].fillna(prep["train_feature_medians"])
    df_any[target_col] = df_any[target_col].fillna(prep["train_target_median"])

    Xs = prep["scaler_X"].transform(df_any[feature_cols])
    out = pd.DataFrame(Xs, columns=feature_cols, index=df_any.index)

    if prep["task_type"] == "classification":
        out[target_col] = df_any[target_col].astype(float).to_numpy()
    else:
        ys = prep["scaler_y"].transform(df_any[[target_col]])
        out[target_col] = ys.flatten()

    return out


def make_windows_no_context(df_scaled, window_size, target_col="Target"):
    if len(df_scaled) <= window_size:
        raise ValueError(
            f"Split too short for window_size={window_size}. "
            f"Need more than {window_size} rows, got {len(df_scaled)}."
        )

    w_df = df_to_windowed_df(df_scaled, window_size=window_size, target_col=target_col)
    dates, X, y = windowed_df_to_date_X_y(w_df, window_size)

    dates = _to_daily_index(np.asarray(dates).reshape(-1))
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    return dates, X, y


def make_windows_with_context(prev_scaled, curr_scaled, window_size, target_col="Target"):
    if len(curr_scaled) == 0:
        raise ValueError("Current split is empty.")

    prev_tail = prev_scaled.tail(window_size).copy()
    combined = pd.concat([prev_tail, curr_scaled], axis=0).copy()
    combined.index = _to_daily_index(combined.index)
    combined = combined.sort_index()

    if len(combined) <= window_size:
        raise ValueError(
            f"Combined split too short for window_size={window_size}. "
            f"Got {len(combined)} rows."
        )

    w_df = df_to_windowed_df(combined, window_size=window_size, target_col=target_col)
    dates, X, y = windowed_df_to_date_X_y(w_df, window_size)

    dates = _to_daily_index(np.asarray(dates).reshape(-1))
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    expected_n = len(curr_scaled)
    curr_dates = _to_daily_index(curr_scaled.index)
    mask = np.asarray(pd.Index(dates).isin(pd.Index(curr_dates)))

    if int(mask.sum()) == expected_n:
        return dates[mask], X[mask], y[mask]

    if len(X) >= expected_n:
        print(
            f"⚠️ Boundary-window fallback used: matched {int(mask.sum())} by date; "
            f"using last {expected_n} windows by position."
        )
        return curr_dates, X[-expected_n:], y[-expected_n:]

    raise ValueError(
        "No boundary-aware windows were created for the current split. "
        f"Expected at least {expected_n}, got {len(X)}."
    )


def split_three_way(df_in, train_end, val_end):
    n_rows = len(df_in)
    i_tr_end = max(1, min(int(n_rows * train_end), n_rows - 2))
    i_val_end = max(i_tr_end + 1, min(int(n_rows * val_end), n_rows - 1))

    df_train = df_in.iloc[:i_tr_end].copy()
    df_val = df_in.iloc[i_tr_end:i_val_end].copy()
    df_test = df_in.iloc[i_val_end:].copy()

    return df_train, df_val, df_test, i_tr_end, i_val_end


def split_two_way(df_in, train_end):
    n_rows = len(df_in)
    i_tr_end = max(1, min(int(n_rows * train_end), n_rows - 1))

    df_train = df_in.iloc[:i_tr_end].copy()
    df_test = df_in.iloc[i_tr_end:].copy()

    return df_train, df_test, i_tr_end


def build_task_model(window_size, n_features, learning_rate, task_type=TASK_TYPE):
    if task_type == "classification":
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(window_size, n_features)),
            tf.keras.layers.LSTM(64),
            tf.keras.layers.Dropout(float(config["lstm"].get("dropout", 0.2))),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss="binary_crossentropy",
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="acc"),
                tf.keras.metrics.AUC(name="auc"),
            ],
        )
        return model

    return build_lstm_model(window_size, n_features, learning_rate)


def eval_split(model, X_split, y_split, scaler_y, ref_split, ticker, out_folder):
    raw_pred = model.predict(X_split, verbose=0).reshape(-1)

    current_price_split = ref_split["Close"].to_numpy(dtype=float)
    true_next_price = ref_split["Target_NextPrice"].to_numpy(dtype=float)

    if TASK_TYPE == "classification":
        y_true = np.asarray(y_split).reshape(-1).astype(int)
        y_prob = raw_pred.astype(float)
        y_hat = (y_prob >= DIRECTION_PROB_THRESHOLD).astype(int)

        dummy_move = max(TRADING_THR, 1e-4)
        pred_next_price = current_price_split * np.where(y_hat == 1, 1.0 + dummy_move, 1.0 - dummy_move)

        acc = float(accuracy_score(y_true, y_hat))
        precision = float(precision_score(y_true, y_hat, zero_division=0))
        recall = float(recall_score(y_true, y_hat, zero_division=0))
        f1 = float(f1_score(y_true, y_hat, zero_division=0))
        auc = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan

        da = compute_directional_accuracy_from_reference(current_price_split, true_next_price, pred_next_price)
        sa = compute_signal_accuracy_from_reference(
            current_price_split,
            true_next_price,
            pred_next_price,
            threshold_pct=TRADING_THR,
            ignore_flat=True,
        )

        pnl_res, trade_df = simulate_pnl(
            np.r_[current_price_split[0], true_next_price],
            np.r_[pred_next_price, pred_next_price[-1]],
            initial_cash=100000,
            ticker=ticker,
            output_folder=out_folder,
            force_first_buy=False,
            threshold_pct=TRADING_THR,
        )

        return {
            "rmse": np.nan,
            "mae": np.nan,
            "mape": np.nan,
            "r2": np.nan,
            "acc": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "auc": auc,
            "da": float(da),
            "sa": float(sa),
            "pnl_ratio": float(pnl_res.get("P&L Ratio", 0.0)),
            "final_cash": float(pnl_res.get("Final Cash", 0.0)),
            "y_true": y_true,
            "y_hat": y_hat,
            "y_prob": y_prob,
            "current_price": current_price_split,
            "pnl_res": pnl_res,
            "trade_df": trade_df,
        }

    y_hat = scaler_y.inverse_transform(raw_pred.reshape(-1, 1)).flatten()
    y_true = scaler_y.inverse_transform(np.asarray(y_split).reshape(-1, 1)).flatten()

    if TARGET_TYPE == "price":
        pred_next_price = y_hat
    elif RETURN_TYPE == "log":
        pred_next_price = current_price_split * np.exp(y_hat)
    else:
        pred_next_price = current_price_split * (1.0 + y_hat)

    rmse, mae, r2, mape = compute_metrics(y_true, y_hat)
    da = compute_directional_accuracy_from_reference(current_price_split, true_next_price, pred_next_price)
    sa = compute_signal_accuracy_from_reference(
        current_price_split,
        true_next_price,
        pred_next_price,
        threshold_pct=TRADING_THR,
        ignore_flat=True,
    )

    pnl_res, trade_df = simulate_pnl(
        np.r_[current_price_split[0], true_next_price],
        np.r_[pred_next_price, pred_next_price[-1]],
        initial_cash=100000,
        ticker=ticker,
        output_folder=out_folder,
        force_first_buy=False,
        threshold_pct=TRADING_THR,
    )

    return {
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2),
        "acc": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "auc": np.nan,
        "da": float(da),
        "sa": float(sa),
        "pnl_ratio": float(pnl_res.get("P&L Ratio", 0.0)),
        "final_cash": float(pnl_res.get("Final Cash", 0.0)),
        "y_true": y_true,
        "y_hat": y_hat,
        "mape": float(mape),
        "current_price": current_price_split,
        "pnl_res": pnl_res,
        "trade_df": trade_df,
    }


def _size_to_rows(size, n_rows):
    size = float(size)
    if 0 < size < 1:
        return max(1, int(np.floor(n_rows * size)))
    return max(1, int(size))


def make_walk_forward_slices(n_rows, min_train_size, val_size, step_size=None, expanding=True):
    train_n = _size_to_rows(min_train_size, n_rows)
    val_n = _size_to_rows(val_size, n_rows)
    step_n = _size_to_rows(step_size if step_size is not None else val_size, n_rows)

    if train_n + val_n > n_rows:
        raise ValueError("Walk-forward setup is too large for the available search block.")

    folds = []
    train_end = train_n

    while train_end + val_n <= n_rows:
        if expanding:
            tr_slice = slice(0, train_end)
        else:
            tr_slice = slice(max(0, train_end - train_n), train_end)
        va_slice = slice(train_end, train_end + val_n)
        folds.append((tr_slice, va_slice))
        train_end += step_n

    if len(folds) == 0:
        raise ValueError("No walk-forward folds were created.")

    return folds


def run_one_fold(df_train, df_val, ref_train, ref_val, config, ticker, *, seed=42, verbose=0, return_artifacts=False):
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    prep = fit_train_only_preprocessor(df_train, target_col="Target", task_type=TASK_TYPE)
    df_train_s = transform_with_preprocessor(df_train, prep, target_col="Target")
    df_val_s = transform_with_preprocessor(df_val, prep, target_col="Target")

    window_size = int(config["lstm"]["window_size"])

    dates_train, X_train, y_train = make_windows_no_context(df_train_s, window_size, target_col="Target")
    dates_val, X_val, y_val = make_windows_with_context(df_train_s, df_val_s, window_size, target_col="Target")

    ref_train_eval = reference_frame_for_dates(ref_train, dates_train)
    ref_val_eval = reference_frame_for_dates(ref_val, dates_val)

    params = config["lstm"]
    model = build_task_model(
        window_size,
        X_train.shape[2],
        float(params["learning_rate"]),
        task_type=TASK_TYPE,
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=int(params["epochs"]),
        batch_size=int(params["batch_size"]),
        verbose=verbose,
        callbacks=callbacks,
    )

    train_m = eval_split(model, X_train, y_train, prep["scaler_y"], ref_train_eval, ticker, "output")
    val_m = eval_split(model, X_val, y_val, prep["scaler_y"], ref_val_eval, ticker, "output")

    out = {
        "n_features": int(X_train.shape[2]),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "train_rmse": train_m["rmse"],
        "train_mae": train_m["mae"],
        "train_r2": train_m["r2"],
        "train_acc": train_m["acc"],
        "train_precision": train_m["precision"],
        "train_mape": train_m["mape"],
        "train_recall": train_m["recall"],
        "train_f1": train_m["f1"],
        "train_auc": train_m["auc"],
        "train_da": train_m["da"],
        "train_sa": train_m["sa"],
        "train_pnl_ratio": train_m["pnl_ratio"],
        "val_rmse": val_m["rmse"],
        "val_mae": val_m["mae"],
        "val_r2": val_m["r2"],
        "val_acc": val_m["acc"],
        "val_precision": val_m["precision"],
        "val_recall": val_m["recall"],
        "val_mape": val_m["mape"],
        "val_f1": val_m["f1"],
        "val_auc": val_m["auc"],
        "val_da": val_m["da"],
        "val_sa": val_m["sa"],
        "val_pnl_ratio": val_m["pnl_ratio"],
    }

    if return_artifacts:
        out.update({
            "dates_train": dates_train,
            "train_true": train_m["y_true"],
            "train_pred": train_m["y_hat"],
            "train_prob": train_m.get("y_prob"),
            "dates_val": dates_val,
            "val_true": val_m["y_true"],
            "val_pred": val_m["y_hat"],
            "val_prob": val_m.get("y_prob"),
            "train_trade_df": train_m["trade_df"],
            "val_trade_df": val_m["trade_df"],
            "train_pnl_res": train_m["pnl_res"],
            "val_pnl_res": val_m["pnl_res"],
            "model": model,
            "scaler_X": prep["scaler_X"],
            "scaler_y": prep["scaler_y"],
            "history": history,
        })

    return out


def run_one_combo(df_work, combo_groups, config, ticker, *, seed=42, verbose=0, return_artifacts=False):
    feature_cols = groups_to_feature_cols(combo_groups)
    if len(feature_cols) == 0:
        raise ValueError("No features available for this combination.")

    df_in = make_numeric_model_df(df_work, feature_cols)
    row_idx = df_in.index.copy()

    df_in = attach_datetime_index(df_in, df_work)
    df_ref = build_reference_frame(df_work, row_idx).reindex(df_in.index)

    sp = config.get("split", {}) or {}
    train_end = float(sp.get("train_end", 0.70))
    val_end = float(sp.get("val_end", 0.80))

    df_train, df_val, _, _, _ = split_three_way(df_in, train_end, val_end)
    ref_train, ref_val, _, _, _ = split_three_way(df_ref, train_end, val_end)

    out = run_one_fold(
        df_train,
        df_val,
        ref_train,
        ref_val,
        config,
        ticker,
        seed=seed,
        verbose=verbose,
        return_artifacts=return_artifacts,
    )

    combo_name = "+".join(combo_groups) if combo_groups else "BASEONLY"
    out.update({
        "combo": combo_name,
        "groups": list(combo_groups),
        "feature_cols": feature_cols,
    })
    return out


def run_one_combo_walk_forward(df_work, combo_groups, config, ticker, *, seed=42, verbose=0):
    feature_cols = groups_to_feature_cols(combo_groups)
    if len(feature_cols) == 0:
        raise ValueError("No features available for this combination.")

    df_in = make_numeric_model_df(df_work, feature_cols)
    row_idx = df_in.index.copy()

    df_in = attach_datetime_index(df_in, df_work)
    df_ref = build_reference_frame(df_work, row_idx).reindex(df_in.index)

    search_end = max(1, min(int(len(df_in) * VAL_END), len(df_in) - 1))
    df_search = df_in.iloc[:search_end].copy()
    ref_search = df_ref.iloc[:search_end].copy()

    folds = make_walk_forward_slices(
        len(df_search),
        min_train_size=WF_MIN_TRAIN_SIZE,
        val_size=WF_VAL_SIZE,
        step_size=WF_STEP_SIZE,
        expanding=WF_EXPANDING,
    )

    fold_rows = []
    for fold_id, (tr_slice, va_slice) in enumerate(folds, start=1):
        fold_out = run_one_fold(
            df_search.iloc[tr_slice].copy(),
            df_search.iloc[va_slice].copy(),
            ref_search.iloc[tr_slice].copy(),
            ref_search.iloc[va_slice].copy(),
            config,
            ticker,
            seed=seed + fold_id,
            verbose=verbose,
            return_artifacts=False,
        )
        fold_out["fold_id"] = fold_id
        fold_rows.append(fold_out)

    fold_df = pd.DataFrame(fold_rows)
    combo_name = "+".join(combo_groups) if combo_groups else "BASEONLY"

    out = {
        "combo": combo_name,
        "groups": list(combo_groups),
        "feature_cols": feature_cols,
        "n_features": int(fold_df["n_features"].iloc[0]),
        "n_folds": int(len(fold_df)),
        "best_epoch": int(np.round(fold_df["best_epoch"].mean())),
    }

    for col in fold_df.columns:
        if col in {"fold_id", "n_features", "best_epoch"}:
            continue
        if pd.api.types.is_numeric_dtype(fold_df[col]):
            out[col] = float(fold_df[col].mean())
            out[f"{col}_std"] = float(fold_df[col].std(ddof=0))

    return out


print("✅ Cell 7 task-aware helpers loaded.")

### Cell 8 — Evaluation helper note

The task-aware evaluation function is defined in Cell 7 together with the preprocessing, windowing, training, and walk-forward helpers.

This notebook keeps Cell 8 as a checkpoint only, so later cells do not accidentally override the active evaluation logic with an older regression-only implementation.

In [ ]:
# ============================================================
# Cell 8 — Evaluation helpers
# Purpose:
# - keep the task-aware eval_split from Cell 7 as the single source of truth
# ============================================================

print("Cell 8 uses the task-aware eval_split defined in Cell 7. No override applied.")

### Cell 9 — Plotting helpers

This cell defines the visualization helper used to compare true and predicted values across the training, validation, and test splits.

The plot provides a qualitative assessment of model fit and generalization. By displaying all three splits together on a shared timeline, it becomes easier to observe whether the model follows the overall price pattern, whether it overfits the training data, and how stable the predictions remain on unseen test data.

The function also allows saving the generated figure to disk for later documentation and inclusion in the thesis.

In [ ]:
# ============================================================
# Cell 9 — Plotting helpers
# Purpose:
# - plot train/val and train/test predictions
# - keep search-stage plots test-free
# ============================================================

def plot_train_val_splits(
    dates_train, y_train_true, y_train_pred,
    dates_val, y_val_true, y_val_pred,
    title="LSTM Predictions (Train/Val)",
    save_path=None,
    ylabel="Next-Day Closing Price"
):
    dates_train = pd.to_datetime(dates_train)
    dates_val = pd.to_datetime(dates_val)

    plt.figure(figsize=(14, 6))
    plt.plot(dates_train, y_train_true, label="Train True")
    plt.plot(dates_train, y_train_pred, label="Train Pred")
    plt.plot(dates_val, y_val_true, label="Val True")
    plt.plot(dates_val, y_val_pred, label="Val Pred")

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)

    plt.show()


def plot_train_test_splits(
    dates_train, y_train_true, y_train_pred,
    dates_test, y_test_true, y_test_pred,
    title="LSTM Predictions (Train/Test)",
    save_path=None,
    ylabel="Next-Day Closing Price"
):
    dates_train = pd.to_datetime(dates_train)
    dates_test = pd.to_datetime(dates_test)

    plt.figure(figsize=(14, 6))
    plt.plot(dates_train, y_train_true, label="Train True")
    plt.plot(dates_train, y_train_pred, label="Train Pred")
    plt.plot(dates_test, y_test_true, label="Test True")
    plt.plot(dates_test, y_test_pred, label="Test Pred")

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)

    plt.show()

### Cell 10 — Search runner setup

This cell activates the validation mode used during the brute-force feature search.

If walk_forward.enable is false, the notebook uses a single chronological train/validation split inside the pre-test block. If walk_forward.enable is true, the notebook uses expanding or rolling walk-forward folds to evaluate each feature combination on repeated validation windows.

The selected models can still be rerun later on a representative single validation split for plotting and artifact saving, but the actual selection itself is based on the configured validation mode.

In [ ]:
# ============================================================
# Cell 10 — Search runner setup
# Purpose:
# - select the active search runner based on config
# - avoid redefining stale price-only experiment code
# ============================================================

RUN_SEARCH_FN = run_one_combo_walk_forward if USE_WALK_FORWARD else run_one_combo
SEARCH_MODE_LABEL = "walk-forward" if USE_WALK_FORWARD else "single-split"

print("✅ Search runner ready:", SEARCH_MODE_LABEL)
print("Representative artifact reruns use the single-split helper after model selection.")

## Cell 11
 

In [ ]:
import itertools

# ============================================================
# Cell 11 — Brute-force experiment
# Purpose:
# - generate all feature combinations
# - run validation search without exposing the final test set
# - support either single-split or walk-forward validation
# - store lightweight summaries only
# ============================================================

search_dir = run_root / "search"
search_dir.mkdir(parents=True, exist_ok=True)

all_groups = list(INDICATOR_GROUPS.keys())

combos = []
if len(BASE_FEATURES) > 0:
    combos.append(())

for r in range(1, len(all_groups) + 1):
    combos.extend(list(itertools.combinations(all_groups, r)))

n_indicator_only_combos = (2 ** len(all_groups)) - 1

print("✅ Total indicator groups:", len(all_groups))
print("✅ Indicator-group non-empty combinations:", n_indicator_only_combos)
print("✅ Total model combinations actually run:", len(combos))
print("✅ BASEONLY included:", len(BASE_FEATURES) > 0)
print("✅ Search mode:", SEARCH_MODE_LABEL)

rows = []

for i, combo in enumerate(combos, start=1):
    combo_label = "+".join(combo) if combo else "BASEONLY"
    print(f"[{i}/{len(combos)}] Running combo: {combo_label}")

    if USE_WALK_FORWARD:
        out = RUN_SEARCH_FN(
            df_work=df_work,
            combo_groups=combo,
            config=config,
            ticker=ticker,
            seed=SEED,
            verbose=0,
        )
    else:
        out = RUN_SEARCH_FN(
            df_work=df_work,
            combo_groups=combo,
            config=config,
            ticker=ticker,
            seed=SEED,
            verbose=0,
            return_artifacts=False,
        )

    rows.append(out)

print("✅ Finished brute-force search.")

df_results = pd.DataFrame(rows)
ranked_results = sort_results_for_selection(df_results).copy()

results_path = search_dir / f"lstm_combo_results_{ticker}_{TARGET_TYPE}.csv"
df_results.to_csv(results_path, index=False)

print("✅ df_results created:", df_results.shape)
print("✅ Saved search results to:", results_path)      
display(df_results.head())
display(ranked_results.head(10))

## Cell 12

In [ ]:
# ============================================================
# Cell 12 — Final-stage baseline helpers
# Purpose:
# - define task-specific baselines
# - defer all test-set baseline evaluation until after model selection
# ============================================================

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def evaluate_final_baseline(df_work, ticker, out_dir):
    baseline_dir = run_root / "baseline"
    baseline_dir.mkdir(parents=True, exist_ok=True)

    n_rows = len(df_work)
    i_test_start = max(1, min(int(n_rows * VAL_END), n_rows - 1))
    df_test_block = df_work.iloc[i_test_start:].copy().reset_index(drop=True)

    if len(df_test_block) == 0:
        raise ValueError("Final baseline evaluation failed because the test block is empty.")

    current_price = df_test_block["Close"].to_numpy(dtype=float)
    true_next_price = df_test_block["Target_NextPrice"].to_numpy(dtype=float)
    baseline_rows = []

    if TARGET_TYPE == "price":
        y_true = df_test_block["Target"].to_numpy(dtype=float)
        y_pred = current_price.copy()

        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        mae = float(mean_absolute_error(y_true, y_pred))
        r2 = float(r2_score(y_true, y_pred))
        mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
        da = float(compute_directional_accuracy_from_reference(current_price, true_next_price, y_pred))
        sa = float(
            compute_signal_accuracy_from_reference(
                current_price,
                true_next_price,
                y_pred,
                threshold_pct=TRADING_THR,
                ignore_flat=True,
            )
        )
        pnl_res, _ = simulate_pnl(
            np.r_[current_price[0], true_next_price],
            np.r_[y_pred, y_pred[-1]],
            initial_cash=100000,
            ticker=ticker,
            output_folder=str(out_dir),
            force_first_buy=False,
            threshold_pct=TRADING_THR,
        )

        baseline_rows.append({
            "model": "Naive Persistence",
            "rmse": rmse,
            "mae": mae,
            "mape": mape,
            "r2": r2,
            "acc": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "da": da,
            "sa": sa,
            "pnl_ratio": float(pnl_res.get("P&L Ratio", 0.0)),
        })

    elif TARGET_TYPE == "return":
        y_true = df_test_block["Target"].to_numpy(dtype=float)
        y_pred = np.zeros_like(y_true, dtype=float)

        if RETURN_TYPE == "log":
            pred_next_price = current_price * np.exp(y_pred)
        else:
            pred_next_price = current_price * (1.0 + y_pred)

        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        mae = float(mean_absolute_error(y_true, y_pred))
        r2 = float(r2_score(y_true, y_pred))
        da = float(compute_directional_accuracy_from_reference(current_price, true_next_price, pred_next_price))
        sa = float(
            compute_signal_accuracy_from_reference(
                current_price,
                true_next_price,
                pred_next_price,
                threshold_pct=TRADING_THR,
                ignore_flat=True,
            )
        )
        pnl_res, _ = simulate_pnl(
            np.r_[current_price[0], true_next_price],
            np.r_[pred_next_price, pred_next_price[-1]],
            initial_cash=100000,
            ticker=ticker,
            output_folder=str(out_dir),
            force_first_buy=False,
            threshold_pct=TRADING_THR,
        )

        baseline_rows.append({
            "model": "Zero-Return Baseline",
            "rmse": rmse,
            "mae": mae,
            "mape": mape,
            "r2": r2,
            "acc": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "da": da,
            "sa": sa,
            "pnl_ratio": float(pnl_res.get("P&L Ratio", 0.0)),
        })

    else:
        y_true = df_test_block["Target"].astype(int).to_numpy()
        prev_direction_full = df_work["Target_Direction"].shift(1)
        if i_test_start > 0:
            fallback_class = int(df_work.iloc[:i_test_start]["Target_Direction"].mode().iloc[0])
        else:
            fallback_class = 1
        y_pred = prev_direction_full.iloc[i_test_start:].fillna(fallback_class).astype(int).to_numpy()

        dummy_move = max(TRADING_THR, 1e-4)
        pred_next_price = current_price * np.where(y_pred == 1, 1.0 + dummy_move, 1.0 - dummy_move)

        acc = float(accuracy_score(y_true, y_pred))
        precision = float(precision_score(y_true, y_pred, zero_division=0))
        recall = float(recall_score(y_true, y_pred, zero_division=0))
        f1 = float(f1_score(y_true, y_pred, zero_division=0))
        da = float(compute_directional_accuracy_from_reference(current_price, true_next_price, pred_next_price))
        sa = float(
            compute_signal_accuracy_from_reference(
                current_price,
                true_next_price,
                pred_next_price,
                threshold_pct=TRADING_THR,
                ignore_flat=True,
            )
        )
        pnl_res, _ = simulate_pnl(
            np.r_[current_price[0], true_next_price],
            np.r_[pred_next_price, pred_next_price[-1]],
            initial_cash=100000,
            ticker=ticker,
            output_folder=str(out_dir),
            force_first_buy=False,
            threshold_pct=TRADING_THR,
        )

        baseline_rows.append({
            "model": "Previous-Direction Baseline",
            "rmse": np.nan,
            "mae": np.nan,
            "mape": np.nan,
            "r2": np.nan,
            "acc": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "auc": np.nan,
            "da": da,
            "sa": sa,
            "pnl_ratio": float(pnl_res.get("P&L Ratio", 0.0)),
        })

    buy_hold_ratio = float((true_next_price[-1] - current_price[0]) / current_price[0])
    baseline_rows.append({
        "model": "Buy-and-Hold",
        "rmse": np.nan,
        "mae": np.nan,
        "r2": np.nan,
        "mape": np.nan,
        "acc": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "auc": np.nan,
        "da": np.nan,
        "sa": np.nan,
        "pnl_ratio": buy_hold_ratio,
    })

    df_baseline = pd.DataFrame(baseline_rows)
    baseline_path = baseline_dir / f"baseline_results_{ticker}_{TARGET_TYPE}.csv"
    df_baseline.to_csv(baseline_path, index=False)

    print("✅ Final baseline saved to:", baseline_path)
    return df_baseline


print("Baseline helper loaded. Final baseline evaluation is deferred until after final model retraining.")

In [ ]:
# ============================================================
# Section 5 — Analytical outputs for thesis (VALIDATION ONLY)
# Purpose:
# - create summary tables and charts from df_results
# - keep the test set hidden until final retraining is complete
# - use task-appropriate ranking metrics
# ============================================================

analysis_dir = run_root / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

ranked_results = sort_results_for_selection(df_results).copy()
primary_metric = "val_f1" if TASK_TYPE == "classification" else "val_rmse"
primary_train_metric = primary_metric.replace("val_", "train_")
primary_label = "Validation F1" if TASK_TYPE == "classification" else "Validation RMSE"
primary_ascending = TASK_TYPE != "classification"

# ------------------------------------------------------------
# 1) Top model tables — validation only
# ------------------------------------------------------------
top_primary_models = ranked_results.head(10).copy()

if TASK_TYPE == "classification":
    top_da_models = df_results.sort_values(
["val_da", "val_f1", "val_pnl_ratio"],
ascending=[False, False, False],
).head(10).copy()
else:
    top_da_models = df_results.sort_values(
["val_da", "val_sa", "val_rmse"],
ascending=[False, False, True],
).head(10).copy()

top_pnl_models = df_results.sort_values(
["val_pnl_ratio", "val_da", primary_metric],
ascending=[False, False, primary_ascending],
).head(10).copy()

#display(top_primary_models)
#display(top_da_models)
#display(top_pnl_models)

print("=" * 70)
print("TABLE 1 — Top 10 Feature Combinations by Primary Metric (Validation)")
print(f" Metric used for ranking: {primary_label}")
print(" Purpose : Feature combination selection — NOT final thesis results.")
print(" These come from walk-forward validation on the first 80% of data.")
print("=" * 70)
display(top_primary_models)

print()
print("=" * 70)
print("TABLE 2 — Top 10 Feature Combinations by Directional Accuracy (Validation)")
print(" Metric used for ranking: Validation DA (how often the model")
print(" predicts the correct next-day price direction).")
print(" Purpose : Identifies the most direction-accurate combination.")
print(" NOT final thesis results — validation stage only.")
print("=" * 70)
display(top_da_models)

print()
print("=" * 70)
print("TABLE 3 — Top 10 Feature Combinations by PnL Ratio (Validation)")
print(" Metric used for ranking: Validation PnL Ratio (avg gain / avg loss)")
print(" per simulated trade on the validation period.")
print(" Purpose : Identifies the most economically useful combination.")
print(" NOT final thesis results — validation stage only.")
print("=" * 70)
display(top_pnl_models)

top_primary_models.to_csv(analysis_dir / f"top_primary_models_{ticker}{TARGET_TYPE}.csv", index=False)
top_da_models.to_csv(analysis_dir / f"top_val_da_models{ticker}{TARGET_TYPE}.csv", index=False)
top_pnl_models.to_csv(analysis_dir / f"top_val_pnl_models{ticker}_{TARGET_TYPE}.csv", index=False)

# ------------------------------------------------------------
# 2) Indicator contribution analysis — validation only
# ------------------------------------------------------------
metric_cols = [
    col for col in [
        "train_rmse", "train_mae", "train_r2", "train_acc", "train_precision", "train_recall", "train_f1", "train_auc",
        "train_da", "train_sa", "train_pnl_ratio",
        "val_rmse", "val_mae", "val_r2", "val_acc", "val_precision", "val_recall", "val_f1", "val_auc",
        "val_da", "val_sa", "val_pnl_ratio",
    ]
    if col in df_results.columns
]

indicator_rows = []
for group in INDICATOR_GROUPS.keys():
    subset = df_results[df_results["combo"].apply(lambda x: combo_has_group(x, group))]
    if len(subset) == 0:
        continue

    row = {
        "indicator": group,
        "n_models": int(len(subset)),
    }
    for col in metric_cols:
        row[f"avg_{col}"] = float(subset[col].mean())
    indicator_rows.append(row)

df_indicator_contrib = pd.DataFrame(indicator_rows)
if not df_indicator_contrib.empty:
    df_indicator_contrib = df_indicator_contrib.sort_values(
        f"avg_{primary_metric}",
        ascending=primary_ascending,
    )

print()
print("=" * 70)
print("TABLE 4 — Indicator Group Contribution (Validation, averaged across")
print(" all feature combinations that include each indicator).")
print(" Metric used for ranking: Average", primary_label)
print(" *** THESIS-RELEVANT — use in Section 3.4.8 / Feature Analysis ***")
print("=" * 70)
display(df_indicator_contrib)
df_indicator_contrib.to_csv(analysis_dir / f"indicator_contribution_{ticker}_{TARGET_TYPE}.csv", index=False)

# ------------------------------------------------------------
# 3) Metric comparison table — validation only
# ------------------------------------------------------------
metric_specs = []
if TASK_TYPE == "classification":
    metric_specs.extend([
        ("Best Validation F1", "val_f1", False),
        ("Best Validation Accuracy", "val_acc", False),
        ("Best Validation AUC", "val_auc", False),
        ("Best Validation DA", "val_da", False),
        ("Best Validation PnL", "val_pnl_ratio", False),
    ])
else:
    metric_specs.extend([
        ("Best Validation RMSE", "val_rmse", True),
        ("Best Validation R2", "val_r2", False),
        ("Best Validation DA", "val_da", False),
        ("Best Validation SA", "val_sa", False),
        ("Best Validation PnL", "val_pnl_ratio", False),
    ])

metric_comparison = pd.DataFrame([
    {
        "metric": label,
        "combo": df_results.sort_values(metric_col, ascending=ascending).iloc[0]["combo"],
        "value": df_results.sort_values(metric_col, ascending=ascending).iloc[0][metric_col],
    }
    for (label, metric_col, ascending) in metric_specs
    if metric_col in df_results.columns
])

print()
print("=" * 70)
print("TABLE 5 — Best Feature Combination per Validation Metric")
print(" Each row shows the winning combination for one evaluation metric.")
print(" *** THESIS-RELEVANT — supports Section 3.4.8 discussion ***")
print("=" * 70)
display(metric_comparison)
metric_comparison.to_csv(analysis_dir / f"metric_comparison_{ticker}_{TARGET_TYPE}.csv", index=False)

# ------------------------------------------------------------
# 4) Generalization gap analysis — train vs validation
# ------------------------------------------------------------
df_generalization = df_results.copy()
gap_metrics = []
for metric in ["rmse", "mae", "r2", "acc", "precision", "recall", "f1", "auc", "da", "sa", "pnl_ratio","mape"]:
    train_col = f"train_{metric}"
    val_col = f"val_{metric}"
    if train_col in df_generalization.columns and val_col in df_generalization.columns:
        gap_col = f"gap_{metric}"
        df_generalization[gap_col] = df_generalization[val_col] - df_generalization[train_col]
        gap_metrics.append(gap_col)

if TASK_TYPE == "classification" and "gap_f1" in df_generalization.columns:
    generalization_summary = df_generalization.sort_values(["gap_f1", "gap_da"], ascending=[False, False]).copy()
elif "gap_rmse" in df_generalization.columns:
    generalization_summary = df_generalization.sort_values(["gap_rmse", "gap_da"], ascending=[True, False]).copy()
else:
    generalization_summary = df_generalization.copy()

print()
print("=" * 70)
print("TABLE 6 — Generalization Gap: Validation minus Training Metrics (Top 15)")
print(" gap_rmse = val_rmse - train_rmse (positive = worse on val = overfitting)")
print(" gap_da = val_da - train_da (positive = better direction on val)")
print(" gap_mape = val_mape - train_mape (positive = worse on val = overfitting)")
print(" Purpose : Overfitting diagnostic — INTERMEDIATE output.")
print("=" * 70)
display(generalization_summary.head(15))
generalization_summary.to_csv(analysis_dir / f"generalization_gap_train_vs_val_{ticker}_{TARGET_TYPE}.csv", index=False)

# ------------------------------------------------------------
# 5) Charts — validation only
# ------------------------------------------------------------
chart_df = top_primary_models.head(10).copy()
plt.figure(figsize=(12, 5))
plt.bar(chart_df["combo"], chart_df[primary_metric])
plt.xticks(rotation=90)
plt.ylabel(primary_label)
plt.title(f"{ticker} — Top 10 Feature Combinations by {primary_label} (Validation)")
plt.tight_layout()
plt.savefig(analysis_dir / f"chart_top_primary_{ticker}_{TARGET_TYPE}.png", dpi=150)
print()
print(f"CHART — Top 10 Feature Combinations by {primary_label} (Validation)")
print(" *** THESIS-RELEVANT — Section 3.4.8 / Feature Combination Analysis ***")
plt.show()

if primary_train_metric in df_results.columns and primary_metric in df_results.columns:
    plt.figure(figsize=(7, 6))
    plt.scatter(df_results[primary_train_metric], df_results[primary_metric])
    plt.xlabel(primary_train_metric.replace("_", " ").title())
    plt.ylabel(primary_metric.replace("_", " ").title())
    plt.title(f"{ticker} — Train vs Validation {primary_label} (All Combinations)")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"chart_train_vs_val_primary_{ticker}_{TARGET_TYPE}.png", dpi=150)
    print()
    print(f"CHART — Train vs Validation {primary_label} Scatter (All 128 Combinations)")
    print(" Purpose : Overfitting diagnostic. Dots near the diagonal generalize well.")
    print(" INTERMEDIATE output — not a primary thesis result.")
    plt.show()







if "val_da" in df_results.columns:
    top_da_plot = df_results.sort_values(["val_da", primary_metric], ascending=[False, primary_ascending]).head(10)
    plt.figure(figsize=(12, 5))
    plt.bar(top_da_plot["combo"], top_da_plot["val_da"])
    plt.xticks(rotation=90)
    plt.ylabel("Validation Directional Accuracy")
    plt.title(f"{ticker} — Top 10 Combinations by Directional Accuracy (Validation)")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"chart_top_val_da_{ticker}_{TARGET_TYPE}.png", dpi=150)
    print()
    print("CHART — Top 10 Combinations by Validation Directional Accuracy")
    print(" DA = how often the model correctly predicts next-day price direction.")
    print(" *** THESIS-RELEVANT — Section 3.7 / Performance Measures ***")
    plt.show()






if not df_indicator_contrib.empty and f"avg_{primary_metric}" in df_indicator_contrib.columns:
    plot_df = df_indicator_contrib.sort_values(f"avg_{primary_metric}", ascending=primary_ascending)
    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["indicator"], plot_df[f"avg_{primary_metric}"])
    plt.ylabel(f"Average {primary_label}")
    plt.title(f"{ticker} — Average {primary_label} by Indicator Group (Validation)")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"chart_indicator_avg_primary_{ticker}_{TARGET_TYPE}.png", dpi=150)
    print()
    print(f"CHART — Average {primary_label} per Indicator Group (Validation)")
    print(" Each bar = average performance across all combinations that include that group.")
    print(" *** THESIS-RELEVANT — Section 3.4.8 / Indicator Contribution Analysis ***")
    plt.show()




if TASK_TYPE == "classification" and "gap_f1" in df_generalization.columns:
    gap_plot = df_generalization.sort_values("val_f1", ascending=False).head(15)
    plt.figure(figsize=(12, 5))
    plt.bar(gap_plot["combo"], gap_plot["gap_f1"])
    plt.xticks(rotation=90)
    plt.ylabel("Validation F1 - Train F1")
    plt.title(f"{ticker} — Generalization Gap: F1 (Validation − Training)")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"chart_generalization_gap_primary_{ticker}{TARGET_TYPE}.png", dpi=150)
    print()
    print("CHART — Generalization Gap: Validation F1 minus Training F1")
    print(" Negative = overfitting. Near-zero = good generalization.")
    print(" INTERMEDIATE diagnostic — not a primary thesis result.")
    plt.show()
elif "gap_rmse" in df_generalization.columns:
    gap_plot = df_generalization.sort_values("val_rmse", ascending=True).head(15)
    plt.figure(figsize=(12, 5))
    plt.bar(gap_plot["combo"], gap_plot["gap_rmse"])
    plt.xticks(rotation=90)
    plt.ylabel("Validation RMSE - Train RMSE")
    plt.title(f"{ticker} — Generalization Gap: RMSE (Validation − Training)")
    plt.tight_layout()
    plt.savefig(analysis_dir / f"chart_generalization_gap_primary{ticker}_{TARGET_TYPE}.png", dpi=150)
    print()
    print("CHART — Generalization Gap: Validation RMSE minus Training RMSE")
    print(" Positive = worse on validation = overfitting. Near-zero = good generalization.")
    print(" INTERMEDIATE diagnostic — not a primary thesis result.")
    plt.show()

print()
print("✅ All validation-only analytical outputs saved to:", analysis_dir)

## Cell 13

In [ ]:
# ============================================================
# Section 6 — Model selection
# Purpose:
# - select models using validation metrics only
# - use task-aware ranking logic
# - rebuild only selected runs for plotting and saving artifacts
# ============================================================

selection_dir = run_root / "model_selection"
selection_dir.mkdir(parents=True, exist_ok=True)

PRIMARY_SELECTION_LABEL = "BEST_VAL_F1" if TASK_TYPE == "classification" else "BEST_VAL_RMSE"

best_val_row = sort_results_for_selection(df_results).iloc[0]
best_val_combo = best_val_row["combo"]

if TASK_TYPE == "classification":
    best_val_da_row = df_results.sort_values(
        ["val_da", "val_f1", "val_pnl_ratio"],
        ascending=[False, False, False],
    ).iloc[0]
else:
    best_val_da_row = df_results.sort_values(
        ["val_da", "val_sa", "val_rmse", "val_r2"],
        ascending=[False, False, True, False],
    ).iloc[0]
best_val_da_combo = best_val_da_row["combo"]

best_val_pnl_row = df_results.sort_values(
    ["val_pnl_ratio", "val_da", primary_metric],
    ascending=[False, False, primary_ascending],
).iloc[0]
best_val_pnl_combo = best_val_pnl_row["combo"]


def build_selection_snapshot(selection_type, row):
    snapshot = {
        "selection_type": selection_type,
        "combo": row["combo"],
        "n_features": row.get("n_features", np.nan),
        "n_folds": row.get("n_folds", np.nan),
    }

    best_epoch = row.get("best_epoch", np.nan)
    snapshot["best_epoch"] = int(round(float(best_epoch))) if pd.notna(best_epoch) else np.nan

    for col in [
        "train_rmse", "train_mae","train_mape", "train_r2", "train_acc", "train_precision", "train_recall", "train_f1", "train_auc",
        "train_da", "train_sa", "train_pnl_ratio",
        "val_rmse", "val_mae", "val_mape", "val_r2", "val_acc", "val_precision", "val_recall", "val_f1", "val_auc",
        "val_da", "val_sa", "val_pnl_ratio",
    ]:
        snapshot[col] = row.get(col, np.nan)

    return snapshot


df_selected_models = pd.DataFrame([
    build_selection_snapshot(PRIMARY_SELECTION_LABEL, best_val_row),
    build_selection_snapshot("BEST_VAL_DA", best_val_da_row),
    build_selection_snapshot("BEST_VAL_PNL", best_val_pnl_row),
])

print("=" * 70)
print("TABLE 11 — Selected Models for Final Retraining (Validation-Based Selection)")
print(" 3 models are carried forward, one per selection criterion:")
print(f" {PRIMARY_SELECTION_LABEL} — best primary metric on validation folds")
print(" BEST_VAL_DA — best directional accuracy on validation folds")
print(" BEST_VAL_PNL — best PnL ratio on validation folds")
print(" All metrics shown are VALIDATION only. Test set not yet evaluated.")
print(" INTERMEDIATE — selection step, not a final result.")
print("=" * 70)
display(df_selected_models)


selected_search_runs = {}
for combo_name in sorted({best_val_combo, best_val_da_combo, best_val_pnl_combo}):
    selected_search_runs[combo_name] = run_one_combo(
        df_work=df_work,
        combo_groups=parse_combo_name(combo_name),
        config=config,
        ticker=ticker,
        seed=SEED,
        verbose=0,
        return_artifacts=True,
    )

best_val_run = selected_search_runs[best_val_combo]
best_val_da_run = selected_search_runs[best_val_da_combo]
best_val_pnl_run = selected_search_runs[best_val_pnl_combo]

print(f"✅ {PRIMARY_SELECTION_LABEL}:", best_val_combo)
print("✅ BEST_VAL_DA:", best_val_da_combo)
print("✅ BEST_VAL_PNL:", best_val_pnl_combo)
if USE_WALK_FORWARD:
    print("Representative validation artifacts were rebuilt on a single split for visualization only.")
else:
    print("Search-stage artifacts were rebuilt from the original single-split validation runs.")


# Section 14 — Presentation outputs for thesis


In [ ]:
# ============================================================
# Section 14 — Presentation outputs for thesis (VALIDATION ONLY)
# Purpose:
# - create thesis-ready tables and charts
# - keep search-stage reporting validation-based
# - use task-appropriate ranking and plots
# ============================================================

presentation_dir = run_root / "presentation"
presentation_dir.mkdir(parents=True, exist_ok=True)

main_result_table = sort_results_for_selection(df_results).copy()
print("=" * 70)
print("TABLE 7 — All Feature Combinations Ranked by Validation Performance")
print(f" Sorted by: {primary_label} (best first).")
print(" Each row = one feature combination and its averaged walk-forward")
print(" validation metrics. TEST SET WAS NOT USED HERE.")
print(" *** THESIS-RELEVANT — Full feature combination search result table. ***")
print(" Saved to: presentation/overall_lstm_results_*.csv")
print("=" * 70)
display(main_result_table.head(15))
main_result_table.to_csv(presentation_dir / f"overall_lstm_results_{ticker}_{TARGET_TYPE}.csv", index=False)



top_models_for_charts = main_result_table.head(15).copy()
chart_specs = [
(primary_metric, primary_label),
("val_da", "Validation Directional Accuracy"),
("val_pnl_ratio", "Validation PnL Ratio"),
("val_mape", "Validation Mean Absolute Percentage Error"),

]
if TASK_TYPE == "classification":
    chart_specs.insert(1, ("val_acc", "Validation Accuracy"))
else:
    chart_specs.insert(1, ("val_sa", "Validation Signal Accuracy"))
    
    
    
for chart_col, chart_label in chart_specs:
    if chart_col not in top_models_for_charts.columns or top_models_for_charts[chart_col].isna().all():
        continue
    plt.figure(figsize=(12, 5))
    plt.bar(top_models_for_charts["combo"], top_models_for_charts[chart_col])
    plt.xticks(rotation=90)
    plt.ylabel(chart_label)
    plt.title(f"{ticker} — Top 15 Combinations: {chart_label} (Validation)")
    plt.tight_layout()
    plt.savefig(presentation_dir / f"{ticker}_{TARGET_TYPE}_{chart_col}.png", dpi=150)
    print()
    print(f"CHART — Top 15 Feature Combinations: {chart_label} (Validation)")
    print(" *** THESIS-RELEVANT — Results chapter, feature combination analysis ***")
    plt.show()

if TASK_TYPE != "direction":
    y_axis_label = "Next-Day Return" if TARGET_TYPE == "return" else "Next-Day Closing Price"
    print()
    print("CHART — Forecast vs Actual: Primary Selected Model on Train/Validation Split")
    print(f"  Model : {PRIMARY_SELECTION_LABEL} | Combination: {best_val_run['combo']}")
    print("  Blue = training period, Orange = validation period.")
    print("  *** THESIS-RELEVANT — Main forecast visualization for selected model ***")
    plot_train_val_splits(
    best_val_run["dates_train"], best_val_run["train_true"], best_val_run["train_pred"],
    best_val_run["dates_val"], best_val_run["val_true"], best_val_run["val_pred"],
    title=f"{ticker} | {PRIMARY_SELECTION_LABEL} | {best_val_run['combo']}",
    save_path=presentation_dir / f"{ticker}_{TARGET_TYPE}_forecast_primary.png",
    ylabel=y_axis_label,
    )

    print()
    print("CHART — Forecast vs Actual: Best DA Model on Train/Validation Split")
    print(f"  Model : BEST_VAL_DA | Combination: {best_val_da_run['combo']}")
    print("  *** THESIS-RELEVANT — Directional accuracy model forecast plot ***")
    plot_train_val_splits(
    best_val_da_run["dates_train"], best_val_da_run["train_true"], best_val_da_run["train_pred"],
    best_val_da_run["dates_val"], best_val_da_run["val_true"], best_val_da_run["val_pred"],
    title=f"{ticker} | BEST_VAL_DA | {best_val_da_run['combo']}",
    save_path=presentation_dir / f"{ticker}_{TARGET_TYPE}_forecast_val_da.png",
    ylabel=y_axis_label,
    )

    print()
    print("CHART — Forecast vs Actual: Best PnL Model on Train/Validation Split")
    print(f"  Model : BEST_VAL_PNL | Combination: {best_val_pnl_run['combo']}")
    print("  *** THESIS-RELEVANT — PnL-selected model forecast plot ***")
    plot_train_val_splits(
    best_val_pnl_run["dates_train"], best_val_pnl_run["train_true"], best_val_pnl_run["train_pred"],
    best_val_pnl_run["dates_val"], best_val_pnl_run["val_true"], best_val_pnl_run["val_pred"],
    title=f"{ticker} | BEST_VAL_PNL | {best_val_pnl_run['combo']}",
    save_path=presentation_dir / f"{ticker}_{TARGET_TYPE}_forecast_val_pnl.png",
    ylabel=y_axis_label,
    )
else:
    print("Direction mode selected: forecast line plots are skipped in favor of classification metrics.")

print()
print("=" * 70)
print("TABLE 8 — Top 15 Feature Combinations (Summary for Thesis)")
print(" Ranked by primary validation metric. Use this in your Results chapter")
print(" table. TEST SET WAS NOT USED.")
print(" *** THESIS-RELEVANT — Table in Results / Feature Combination Analysis ***")
print("=" * 70)
top_combinations_table = main_result_table.head(15).copy()
display(top_combinations_table)
top_combinations_table.to_csv(presentation_dir / f"top_feature_combinations_{ticker}_{TARGET_TYPE}.csv", index=False)

print()
print("=" * 70)
print("TABLE 8 — Top 15 Feature Combinations (Summary for Thesis)")
print(" Ranked by primary validation metric. Use this in your Results chapter")
print(" table. TEST SET WAS NOT USED.")
print(" *** THESIS-RELEVANT — Table in Results / Feature Combination Analysis ***")
print("=" * 70)
top_combinations_table = main_result_table.head(15).copy()
display(top_combinations_table)
top_combinations_table.to_csv(presentation_dir / f"top_feature_combinations_{ticker}_{TARGET_TYPE}.csv", index=False)


top_n = 20
top_subset = main_result_table.head(top_n).copy()
groups = list(INDICATOR_GROUPS.keys())
co_matrix = pd.DataFrame(0, index=groups, columns=groups, dtype=int)
for combo in top_subset["combo"]:
    parts = parse_combo_name(combo)
    for g1 in parts:
        for g2 in parts:
            if g1 in co_matrix.index and g2 in co_matrix.columns:
                co_matrix.loc[g1, g2] += 1


print()
print("=" * 70)
print(f"TABLE 10 — Indicator Co-occurrence Matrix (Top {top_n} Combinations)")
print(" Cell [i, j] = how many top-20 combinations include BOTH group i and group j.")
print(" Diagonal = frequency of each individual group (same as Table 9).")
print(" *** THESIS-RELEVANT — Feature combination interaction analysis ***")
print("=" * 70)
display(co_matrix)
co_matrix.to_csv(presentation_dir / f"indicator_cooccurrence_{ticker}_{TARGET_TYPE}.csv")

plt.figure(figsize=(8, 6))
plt.imshow(co_matrix, aspect="auto")
plt.colorbar(label="Co-occurrence Count")
plt.xticks(range(len(groups)), groups, rotation=45)
plt.yticks(range(len(groups)), groups)
plt.title(f"{ticker} — Indicator Co-occurrence Heatmap (Top {top_n} Validation-Ranked Combinations)")
plt.tight_layout()
plt.savefig(presentation_dir / f"{ticker}_{TARGET_TYPE}_indicator_cooccurrence_heatmap.png", dpi=150)
print()
print(f"CHART — Indicator Co-occurrence Heatmap (Top {top_n} Combinations)")
print(" Bright cells = those two indicators frequently appear together in top models.")
print(" *** THESIS-RELEVANT — Feature interaction visualization ***")
plt.show()

## Cell 15 

In [ ]:
# ============================================================
# Cell 15 — Final retraining on TRAIN+VAL and evaluation on TEST
# Purpose:
# - rebuild the selected validation-based models
# - train on the full pre-test block (TRAIN+VAL)
# - evaluate once on the untouched TEST set
# - compare against a final-stage baseline
# ============================================================

final_dir = run_root / "final"
final_dir.mkdir(parents=True, exist_ok=True)
PRIMARY_FINAL_LABEL = f"FINAL_{PRIMARY_SELECTION_LABEL}"


def prepare_final_train_test(df_work, feature_cols, config, *, train_end_frac):
    df_in = make_numeric_model_df(df_work, feature_cols)
    row_idx = df_in.index.copy()

    df_in = attach_datetime_index(df_in, df_work)
    df_ref = build_reference_frame(df_work, row_idx).reindex(df_in.index)

    df_train, df_test, _ = split_two_way(df_in, train_end_frac)
    ref_train, ref_test, _ = split_two_way(df_ref, train_end_frac)

    prep = fit_train_only_preprocessor(df_train, target_col="Target", task_type=TASK_TYPE)
    df_train_s = transform_with_preprocessor(df_train, prep, target_col="Target")
    df_test_s = transform_with_preprocessor(df_test, prep, target_col="Target")

    window_size = int(config["lstm"]["window_size"])

    dates_train, X_train, y_train = make_windows_no_context(df_train_s, window_size, target_col="Target")
    dates_test, X_test, y_test = make_windows_with_context(df_train_s, df_test_s, window_size, target_col="Target")

    ref_train_eval = reference_frame_for_dates(ref_train, dates_train)
    ref_test_eval = reference_frame_for_dates(ref_test, dates_test)

    return (
        X_train,
        y_train,
        dates_train,
        X_test,
        y_test,
        dates_test,
        ref_train_eval,
        ref_test_eval,
        prep["scaler_X"],
        prep["scaler_y"],
        window_size,
    )


def retrain_selected_model(df_work, combo_name, config, ticker, *, train_end_frac, epochs_fixed, seed=42, verbose=1):
    tf.keras.utils.set_random_seed(seed)
    tf.keras.backend.clear_session()

    groups = parse_combo_name(combo_name)
    feature_cols = groups_to_feature_cols(groups)

    (
        X_train,
        y_train,
        dates_train,
        X_test,
        y_test,
        dates_test,
        ref_train_eval,
        ref_test_eval,
        scaler_X,
        scaler_y,
        window_size,
    ) = prepare_final_train_test(
        df_work,
        feature_cols,
        config,
        train_end_frac=train_end_frac,
    )

    params = config["lstm"]
    model = build_task_model(
        window_size,
        X_train.shape[2],
        float(params["learning_rate"]),
        task_type=TASK_TYPE,
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=int(epochs_fixed),
        batch_size=int(params["batch_size"]),
        verbose=verbose,
    )

    train_m = eval_split(model, X_train, y_train, scaler_y, ref_train_eval, ticker, str(final_dir))
    test_m = eval_split(model, X_test, y_test, scaler_y, ref_test_eval, ticker, str(final_dir))

    return {
        "combo": combo_name,
        "groups": groups,
        "feature_cols": feature_cols,
        "n_features": int(X_train.shape[2]),
        "epochs_used": int(epochs_fixed),
        "X_train": X_train,
        "y_train_scaled": y_train,
        "X_test": X_test,
        "y_test_scaled": y_test,
        "train_rmse": train_m["rmse"],
        "train_mae": train_m["mae"],
        "train_mape": train_m["mape"],
        "train_r2": train_m["r2"],
        "train_acc": train_m["acc"],
        "train_precision": train_m["precision"],
        "train_recall": train_m["recall"],
        "train_f1": train_m["f1"],
        "train_auc": train_m["auc"],
        "train_da": train_m["da"],
        "train_sa": train_m["sa"],
        "train_pnl_ratio": train_m["pnl_ratio"],
        "test_rmse": test_m["rmse"],
        "test_mae": test_m["mae"],
        "test_r2": test_m["r2"],
        "test_acc": test_m["acc"],
        "test_precision": test_m["precision"],
        "test_recall": test_m["recall"],
        "test_f1": test_m["f1"],
        "test_auc": test_m["auc"],
        "test_da": test_m["da"],
        "test_sa": test_m["sa"],
        "test_pnl_ratio": test_m["pnl_ratio"],
        "test_mape": test_m["mape"],
        "dates_train": dates_train,
        "train_true": train_m["y_true"],
        "train_pred": train_m["y_hat"],
        "train_prob": train_m.get("y_prob"),
        "dates_test": dates_test,
        "test_true": test_m["y_true"],
        "test_pred": test_m["y_hat"],
        "test_prob": test_m.get("y_prob"),
        "train_current_price": ref_train_eval["Close"].to_numpy(dtype=float),
        "test_current_price": ref_test_eval["Close"].to_numpy(dtype=float),
        "train_trade_df": train_m["trade_df"],
        "test_trade_df": test_m["trade_df"],
        "train_pnl_res": train_m["pnl_res"],
        "test_pnl_res": test_m["pnl_res"],
        "model": model,
        "history": history,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y,
    }


final_train_end = float(config.get("split", {}).get("val_end", 0.80))
epochs_best_primary = int(round(float(best_val_row["best_epoch"])))
epochs_best_val_da = int(round(float(best_val_da_row["best_epoch"])))
epochs_best_val_pnl = int(round(float(best_val_pnl_row["best_epoch"])))

print("Final retraining split end (TRAIN+VAL):", final_train_end)
print(f"Epochs for {PRIMARY_SELECTION_LABEL}:", epochs_best_primary)
print("Epochs for BEST_VAL_DA:", epochs_best_val_da)
print("Epochs for BEST_VAL_PNL:", epochs_best_val_pnl)

final_run_cache = {}


def get_final_run(combo_name, epochs_fixed):
    key = (combo_name, int(epochs_fixed))
    if key not in final_run_cache:
        final_run_cache[key] = retrain_selected_model(
            df_work=df_work,
            combo_name=combo_name,
            config=config,
            ticker=ticker,
            train_end_frac=final_train_end,
            epochs_fixed=int(epochs_fixed),
            seed=SEED,
            verbose=1,
        )
    return final_run_cache[key]


final_best_val = get_final_run(best_val_combo, epochs_best_primary)
final_best_val_da = get_final_run(best_val_da_combo, epochs_best_val_da)
final_best_val_pnl = get_final_run(best_val_pnl_combo, epochs_best_val_pnl)

FINAL_RUNS = {
    PRIMARY_FINAL_LABEL: final_best_val,
    "FINAL_BEST_VAL_DA": final_best_val_da,
    "FINAL_BEST_VAL_PNL": final_best_val_pnl,
}


def build_final_snapshot(selection_type, run_obj):
    snapshot = {
        "selection_type": selection_type,
        "combo": run_obj["combo"],
        "n_features": run_obj["n_features"],
        "epochs_used": run_obj["epochs_used"],
    }
    for col in [
        "train_rmse", "train_mae","train_mape", "train_r2", "train_acc", "train_precision", "train_recall", "train_f1", "train_auc",
        "train_da", "train_sa", "train_pnl_ratio",
        "test_rmse", "test_mae", "test_mape", "test_r2", "test_acc", "test_precision", "test_recall", "test_f1", "test_auc",
        "test_da", "test_sa", "test_pnl_ratio",
    ]:
        snapshot[col] = run_obj.get(col, np.nan)
    return snapshot


df_final_models = pd.DataFrame([
    build_final_snapshot(PRIMARY_FINAL_LABEL, final_best_val),
    build_final_snapshot("FINAL_BEST_VAL_DA", final_best_val_da),
    build_final_snapshot("FINAL_BEST_VAL_PNL", final_best_val_pnl),
])

display(df_final_models)
df_final_models.to_csv(final_dir / f"final_retrained_models_{ticker}_{TARGET_TYPE}.csv", index=False)
print("✅ Saved:", final_dir / f"final_retrained_models_{ticker}_{TARGET_TYPE}.csv")

df_baseline = evaluate_final_baseline(df_work, ticker, final_dir)
display(df_baseline)

## Cell 16

In [ ]:
# ============================================================
# Cell 16 — Unified official saver
# Purpose:
# - save all official artifacts in one place
# - keep search-stage outputs validation-only
# - keep final-stage outputs test-only
# - support price, return, and direction tasks
# ============================================================

import json
import yaml
import joblib

RUN_ROOT = run_root
ARTIFACTS = {
    "root": RUN_ROOT,
    "search": RUN_ROOT / "search",
    "final": RUN_ROOT / "final",
    "tables": RUN_ROOT / "tables",
    "plots": RUN_ROOT / "plots",
    "models": RUN_ROOT / "models",
    "scalers": RUN_ROOT / "scalers",
    "histories": RUN_ROOT / "histories",
    "logs": RUN_ROOT / "logs",
    "pnl_logs": RUN_ROOT / "pnl_logs",
    "shap": RUN_ROOT / "shap",
}

for p in ARTIFACTS.values():
    p.mkdir(parents=True, exist_ok=True)


def to_jsonable(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, pd.Timestamp):
        return x.isoformat()
    if isinstance(x, pd.Series):
        return x.tolist()
    if isinstance(x, pd.DataFrame):
        return x.to_dict(orient="records")
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_jsonable(v) for v in x]
    return x


def save_run_bundle(run_obj, tag, *, stage):
    combo = run_obj["combo"]
    prefix = f"{stage.upper()}_{tag}_{combo}"

    run_obj["model"].save(ARTIFACTS["models"] / f"{prefix}.keras")
    joblib.dump(run_obj["scaler_X"], ARTIFACTS["scalers"] / f"{prefix}_scaler_X.pkl")
    if run_obj.get("scaler_y") is not None:
        joblib.dump(run_obj["scaler_y"], ARTIFACTS["scalers"] / f"{prefix}_scaler_y.pkl")

    hist_df = pd.DataFrame(run_obj["history"].history)
    hist_df.to_csv(ARTIFACTS["histories"] / f"{prefix}_history.csv", index=False)

    summary = {
        "ticker": ticker,
        "target_type": TARGET_TYPE,
        "task_type": TASK_TYPE,
        "stage": stage,
        "tag": tag,
        "combo": combo,
        "groups": run_obj.get("groups"),
        "feature_cols": run_obj.get("feature_cols"),
        "n_features": run_obj.get("n_features"),
        "epochs_used": run_obj.get("epochs_used", run_obj.get("best_epoch")),
        "train_rmse": run_obj.get("train_rmse"),
        "train_mae": run_obj.get("train_mae"),
        "train_r2": run_obj.get("train_r2"),
        "train_acc": run_obj.get("train_acc"),
        "train_precision": run_obj.get("train_precision"),
        "train_recall": run_obj.get("train_recall"),
        "train_f1": run_obj.get("train_f1"),
        "train_auc": run_obj.get("train_auc"),
        "train_da": run_obj.get("train_da"),
        "train_sa": run_obj.get("train_sa"),
        "train_pnl_ratio": run_obj.get("train_pnl_ratio"),
        "val_rmse": run_obj.get("val_rmse"),
        "val_mae": run_obj.get("val_mae"),
        "val_r2": run_obj.get("val_r2"),
        "val_acc": run_obj.get("val_acc"),
        "val_precision": run_obj.get("val_precision"),
        "val_recall": run_obj.get("val_recall"),
        "val_f1": run_obj.get("val_f1"),
        "val_auc": run_obj.get("val_auc"),
        "val_da": run_obj.get("val_da"),
        "val_sa": run_obj.get("val_sa"),
        "val_pnl_ratio": run_obj.get("val_pnl_ratio"),
        "test_rmse": run_obj.get("test_rmse"),
        "test_mae": run_obj.get("test_mae"),
        "test_r2": run_obj.get("test_r2"),
        "test_acc": run_obj.get("test_acc"),
        "test_precision": run_obj.get("test_precision"),
        "test_recall": run_obj.get("test_recall"),
        "test_f1": run_obj.get("test_f1"),
        "test_auc": run_obj.get("test_auc"),
        "test_da": run_obj.get("test_da"),
        "test_sa": run_obj.get("test_sa"),
        "test_pnl_ratio": run_obj.get("test_pnl_ratio"),
        "train_pnl_res": run_obj.get("train_pnl_res", {}),
        "val_pnl_res": run_obj.get("val_pnl_res", {}),
        "test_pnl_res": run_obj.get("test_pnl_res", {}),
    }

    with open(ARTIFACTS["logs"] / f"{prefix}_summary.json", "w") as f:
        json.dump(to_jsonable(summary), f, indent=2)

    if TARGET_TYPE != "direction":
        ylabel = "Next-Day Return" if TARGET_TYPE == "return" else "Next-Day Closing Price"
        if stage == "search":
            plot_train_val_splits(
                run_obj["dates_train"], run_obj["train_true"], run_obj["train_pred"],
                run_obj["dates_val"], run_obj["val_true"], run_obj["val_pred"],
                title=f"{ticker} | SEARCH | {tag} | {combo}",
                save_path=ARTIFACTS["plots"] / f"{prefix}_predictions.png",
                ylabel=ylabel,
            )
        else:
            plot_train_test_splits(
                run_obj["dates_train"], run_obj["train_true"], run_obj["train_pred"],
                run_obj["dates_test"], run_obj["test_true"], run_obj["test_pred"],
                title=f"{ticker} | FINAL | {tag} | {combo}",
                save_path=ARTIFACTS["plots"] / f"{prefix}_predictions.png",
                ylabel=ylabel,
            )

    if run_obj.get("train_trade_df") is not None:
        run_obj["train_trade_df"].to_csv(ARTIFACTS["pnl_logs"] / f"{prefix}_TRAIN_pnl_log.csv", index=False)
    if run_obj.get("val_trade_df") is not None:
        run_obj["val_trade_df"].to_csv(ARTIFACTS["pnl_logs"] / f"{prefix}_VAL_pnl_log.csv", index=False)
    if run_obj.get("test_trade_df") is not None:
        run_obj["test_trade_df"].to_csv(ARTIFACTS["pnl_logs"] / f"{prefix}_TEST_pnl_log.csv", index=False)


def save_official_artifacts():
    with open(ARTIFACTS["root"] / "config.yaml", "w") as f:
        yaml.dump(config, f, sort_keys=False)

    df_results.to_csv(ARTIFACTS["tables"] / f"results_{TARGET_TYPE}.csv", index=False)
    df_selected_models.to_csv(ARTIFACTS["tables"] / f"selected_models_{TARGET_TYPE}.csv", index=False)
    df_final_models.to_csv(ARTIFACTS["tables"] / f"final_retrained_models_{TARGET_TYPE}.csv", index=False)
    if "df_baseline" in globals() and df_baseline is not None:
        df_baseline.to_csv(ARTIFACTS["tables"] / f"baseline_results_{TARGET_TYPE}.csv", index=False)
    if "df_indicator_contrib" in globals() and df_indicator_contrib is not None and len(df_indicator_contrib) > 0:
        df_indicator_contrib.to_csv(ARTIFACTS["tables"] / f"indicator_contribution_{TARGET_TYPE}.csv", index=False)

    run_logs = {
        "ticker": ticker,
        "target_type": TARGET_TYPE,
        "task_type": TASK_TYPE,
        "search_mode": SEARCH_MODE_LABEL,
        "n_models_trained": int(len(df_results)),
        "split": {
            "train_end": TRAIN_END,
            "val_end": VAL_END,
        },
        "walk_forward": {
            "enabled": USE_WALK_FORWARD,
            "expanding": WF_EXPANDING,
            "min_train_size": WF_MIN_TRAIN_SIZE,
            "val_size": WF_VAL_SIZE,
            "step_size": WF_STEP_SIZE,
        },
        "primary_selection_label": PRIMARY_SELECTION_LABEL,
        "selected_search_models": to_jsonable(df_selected_models.to_dict(orient="records")),
        "final_models": to_jsonable(df_final_models.to_dict(orient="records")),
        "baseline": to_jsonable(df_baseline.to_dict(orient="records")) if "df_baseline" in globals() else None,
    }

    with open(ARTIFACTS["logs"] / "run_logs.json", "w") as f:
        json.dump(to_jsonable(run_logs), f, indent=2)

    save_run_bundle(best_val_run, PRIMARY_SELECTION_LABEL, stage="search")
    save_run_bundle(best_val_da_run, "BEST_VAL_DA", stage="search")
    save_run_bundle(best_val_pnl_run, "BEST_VAL_PNL", stage="search")
    save_run_bundle(final_best_val, PRIMARY_FINAL_LABEL, stage="final")
    save_run_bundle(final_best_val_da, "FINAL_BEST_VAL_DA", stage="final")
    save_run_bundle(final_best_val_pnl, "FINAL_BEST_VAL_PNL", stage="final")

    print("✅ Unified official artifacts saved to:", ARTIFACTS["root"])


save_official_artifacts()

# Cell 17 - using shap for test

In [ ]:
import shap

# ============================================================
# Section 7A.2 — SHAP helpers for LSTM
# ============================================================

def compute_lstm_shap(run_obj, n_background=50, n_explain=20):
    """
    Compute SHAP values for a trained LSTM model.

    Parameters
    ----------
    run_obj : dict
        Final trained run object from train_test_no_val().
    n_background : int
        Number of background samples taken from X_train.
    n_explain : int
        Number of test samples to explain.

    Returns
    -------
    shap_values_arr : np.ndarray
        SHAP values for the explained test samples.
        Expected shape: (samples, timesteps, features)
    X_explain : np.ndarray
        Test samples used for explanation.
    X_background : np.ndarray
        Background train samples.
    """
    model = run_obj["model"]
    X_train = run_obj["X_train"]
    X_test = run_obj["X_test"]

    # Small background subset
    X_background = X_train[:min(n_background, len(X_train))]

    # Small explain subset
    X_explain = X_test[:min(n_explain, len(X_test))]

    # GradientExplainer is usually the best choice for Keras LSTM
    explainer = shap.GradientExplainer(model, X_background)
    shap_values = explainer.shap_values(X_explain)

    # Handle different SHAP output formats
    if isinstance(shap_values, list):
        shap_values_arr = shap_values[0]
    else:
        shap_values_arr = shap_values

    # If output contains final singleton dimension, remove it
    if len(shap_values_arr.shape) == 4 and shap_values_arr.shape[-1] == 1:
        shap_values_arr = shap_values_arr[..., 0]

    return shap_values_arr, X_explain, X_background

## Cell 18 it continiues 

In [ ]:
def plot_shap_global_importance(run_obj, shap_values_arr, save_path=None, top_n=None):
    """
    Plot global SHAP importance aggregated across samples and timesteps.
    """
    feature_cols = run_obj["feature_cols"]

    # shap_values_arr shape: (samples, timesteps, features)
    mean_abs_shap = np.abs(shap_values_arr).mean(axis=(0, 1))

    df_imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    if top_n is not None:
        df_imp = df_imp.head(top_n)

    plt.figure(figsize=(10, 5))
    plt.bar(df_imp["feature"], df_imp["mean_abs_shap"])
    plt.xticks(rotation=90)
    plt.ylabel("Mean |SHAP value|")
    plt.title("Global SHAP Feature Importance")
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)

    plt.show()

    return df_imp

## Cell 19

In [ ]:
def plot_shap_local_explanation(run_obj, shap_values_arr, X_explain, sample_idx=0, save_path=None):
    """
    Plot local SHAP explanation for one test sample.
    Aggregates SHAP over timesteps so each feature has one contribution.
    """
    feature_cols = run_obj["feature_cols"]

    # One sample: shape (timesteps, features)
    sample_shap = shap_values_arr[sample_idx]

    # Aggregate over time
    feature_contrib = sample_shap.sum(axis=0)

    df_local = pd.DataFrame({
        "feature": feature_cols,
        "shap_contribution": feature_contrib
    }).sort_values("shap_contribution", key=np.abs, ascending=False)

    plt.figure(figsize=(10, 5))
    colors = ["#ff0051" if v > 0 else "#118dff" for v in df_local["shap_contribution"]]
    plt.barh(df_local["feature"], df_local["shap_contribution"], color=colors)
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("SHAP contribution")
    plt.ylabel("Feature")
    plt.title(f"Local SHAP Explanation (sample {sample_idx})")
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)

    plt.show()

    return df_local

## Cell 20

In [ ]:
# ============================================================
# Section 7A.3 — Run SHAP on selected final models
# ============================================================

shap_dir = run_root / "shap"
shap_dir.mkdir(parents=True, exist_ok=True)

shap_values_val, X_explain_val, X_background_val = compute_lstm_shap(
    final_best_val,
    n_background=50,
    n_explain=20,
)

df_shap_global_val = plot_shap_global_importance(
    final_best_val,
    shap_values_val,
    save_path=shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_PRIMARY_global_shap.png",
)

df_shap_local_val = plot_shap_local_explanation(
    final_best_val,
    shap_values_val,
    X_explain_val,
    sample_idx=0,
    save_path=shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_PRIMARY_local_shap_sample0.png",
)

df_shap_global_val.to_csv(shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_PRIMARY_global_shap.csv", index=False)
df_shap_local_val.to_csv(shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_PRIMARY_local_shap_sample0.csv", index=False)

In [ ]:
shap_values_da, X_explain_da, X_background_da = compute_lstm_shap(
    final_best_val_da,
    n_background=50,
    n_explain=20,
)

df_shap_global_da = plot_shap_global_importance(
    final_best_val_da,
    shap_values_da,
    save_path=shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_BEST_VAL_DA_global_shap.png",
)

df_shap_local_da = plot_shap_local_explanation(
    final_best_val_da,
    shap_values_da,
    X_explain_da,
    sample_idx=0,
    save_path=shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_BEST_VAL_DA_local_shap_sample0.png",
)

df_shap_global_da.to_csv(
    shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_BEST_VAL_DA_global_shap.csv",
    index=False,
)

df_shap_local_da.to_csv(
    shap_dir / f"{ticker}_{TARGET_TYPE}_FINAL_BEST_VAL_DA_local_shap_sample0.csv",
    index=False,
)

## Cell 23

In [ ]:
# ------------------------------------------------------------
# Print final evaluation summaries
# ------------------------------------------------------------

summary_metric_cols = [
    ("Test F1", "test_f1"),
    ("Test Accuracy", "test_acc"),
    ("Test AUC", "test_auc"),
    ("Test DA", "test_da"),
    ("Test PnL", "test_pnl_ratio"),
] if TASK_TYPE == "classification" else [
    ("Test RMSE", "test_rmse"),
    ("Test MAE", "test_mae"),
    ("Test R2", "test_r2"),
    ("Test DA", "test_da"),
    ("Test SA", "test_sa"),
    ("Test PnL", "test_pnl_ratio"),
]

for name, run_obj in [
    (PRIMARY_FINAL_LABEL, final_best_val),
    ("FINAL_BEST_VAL_DA", final_best_val_da),
    ("FINAL_BEST_VAL_PNL", final_best_val_pnl),
]:
    print(f"✅ {name}: {run_obj['combo']}")
    for label, key in summary_metric_cols:
        print(f"   {label}: {run_obj.get(key, np.nan)}")
    print()

if "df_baseline" in globals() and df_baseline is not None:
    print("✅ Final baseline comparison")
    display(df_baseline)

## Cell 25

In [ ]:
# ------------------------------------------------------------
# Final comparison charts
# ------------------------------------------------------------

primary_test_metric = "test_f1" if TASK_TYPE == "classification" else "test_rmse"
primary_test_label = "Test F1" if TASK_TYPE == "classification" else "Test RMSE"

if primary_test_metric in df_final_models.columns and not df_final_models[primary_test_metric].isna().all():
    plt.figure(figsize=(10, 5))
    plt.bar(df_final_models["selection_type"], df_final_models[primary_test_metric])
    plt.ylabel(primary_test_label)
    plt.title(f"{ticker} — Final Retrained Models: {primary_test_label}")
    plt.tight_layout()
    plt.savefig(final_dir / f"final_models_{TARGET_TYPE}_{primary_test_metric}_{ticker}.png", dpi=150)
    plt.show()

plt.figure(figsize=(10, 5))
plt.bar(df_final_models["selection_type"], df_final_models["test_da"])
plt.ylabel("Test Directional Accuracy")
plt.title(f"{ticker} — Final Retrained Models: Test DA")
plt.tight_layout()
plt.savefig(final_dir / f"final_models_{TARGET_TYPE}_test_da_{ticker}.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(df_final_models["selection_type"], df_final_models["test_pnl_ratio"])
plt.ylabel("Test PnL Ratio")
plt.title(f"{ticker} — Final Retrained Models: Test PnL Ratio")
plt.tight_layout()
plt.savefig(final_dir / f"final_models_{TARGET_TYPE}_test_pnl_{ticker}.png", dpi=150)
plt.show()

if "df_baseline" in globals() and df_baseline is not None and "pnl_ratio" in df_baseline.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(df_baseline["model"], df_baseline["pnl_ratio"])
    plt.ylabel("PnL Ratio")
    plt.title(f"{ticker} — Final Baseline Comparison")
    plt.tight_layout()
    plt.savefig(final_dir / f"baseline_{TARGET_TYPE}_pnl_{ticker}.png", dpi=150)
    plt.show()

## Cell 26 — Load saved results and analyse selected models

This cell loads all artifacts produced by the pipeline and produces a comprehensive analysis:
- (A) Full combo search overview and ranking
- (B) Selected models' validation metrics (walk-forward averages)
- (C) Final retrained models' test-set metrics
- (D) Overfitting gap (train vs. test)
- (E) LSTM vs. baselines comparison
- (F) Training loss curves
- (G) Prediction residual analysis (scatter, time series, histogram)
- (H) Trade log analysis

In [ ]:
# ============================================================
# Cell 26 — Load saved results and analyse selected models
# ============================================================
import glob, json
from pathlib import Path

_tables   = run_root / "tables"
_logs     = run_root / "logs"
_hist     = run_root / "histories"
_pnl_dir  = run_root / "pnl_logs"

# ── load tables ────────────────────────────────────────────
df_results   = pd.read_csv(_tables / f"results_{TARGET_TYPE}.csv")
df_selected  = pd.read_csv(_tables / f"selected_models_{TARGET_TYPE}.csv")
df_final     = pd.read_csv(_tables / f"final_retrained_models_{TARGET_TYPE}.csv")
df_baseline  = pd.read_csv(_tables / f"baseline_results_{TARGET_TYPE}.csv")

# ===========================================================
# A — Combo Search Overview
# ===========================================================
print(f"{'='*60}")
print(f"  A. COMBO SEARCH OVERVIEW  ({ticker} / {TARGET_TYPE})")
print(f"{'='*60}")
print(f"  Total combos evaluated : {len(df_results)}")
print(f"  Features per combo     : {df_results['n_features'].min()}–{df_results['n_features'].max()}")
print(f"  Folds (walk-forward)   : {int(df_results['n_folds'].iloc[0])}")
print()

# primary sort column is the first in SEARCH_SORT_COLS
_primary = SEARCH_SORT_COLS[0]
_asc     = SEARCH_SORT_ASC[0]
df_sorted = df_results.sort_values(SEARCH_SORT_COLS, ascending=SEARCH_SORT_ASC).reset_index(drop=True)

_show_cols = ["combo", "n_features"] + SEARCH_SORT_COLS
_show_cols = [c for c in _show_cols if c in df_sorted.columns]

print(f"  Top 10 combos by [{', '.join(SEARCH_SORT_COLS)}] ({'ascending' if _asc else 'descending'} primary):")
display(df_sorted[_show_cols].head(10).round(4))

# ===========================================================
# B — Selected Models — Validation Metrics
# ===========================================================
print(f"\n{'='*60}")
print("  B. SELECTED MODELS — VALIDATION METRICS (walk-forward avg)")
print(f"{'='*60}")

_val_show = ["selection_type", "combo", "n_features", "n_folds", "best_epoch",
             "val_rmse", "val_r2", "val_da", "val_sa", "val_pnl_ratio"]
_val_show = [c for c in _val_show if c in df_selected.columns]
display(df_selected[_val_show].round(4))

# ===========================================================
# C — Final Retrained Models — Test-Set Metrics
# ===========================================================
print(f"\n{'='*60}")
print("  C. FINAL RETRAINED MODELS — TEST-SET METRICS")
print(f"{'='*60}")

_fin_show = ["selection_type", "combo", "n_features", "epochs_used",
             "train_rmse", "train_da", "train_pnl_ratio",
             "test_rmse",  "test_da",  "test_pnl_ratio"]
_fin_show = [c for c in _fin_show if c in df_final.columns]
display(df_final[_fin_show].round(4))

# ===========================================================
# D — Overfitting Gap
# ===========================================================
print(f"\n{'='*60}")
print("  D. OVERFITTING GAP  (test - train for RMSE;  train - test for DA)")
print(f"{'='*60}")

df_gap = df_final[["selection_type", "combo"]].copy()
if "train_rmse" in df_final.columns and "test_rmse" in df_final.columns:
    df_gap["gap_rmse"] = (df_final["test_rmse"] - df_final["train_rmse"]).round(4)
if "train_da" in df_final.columns and "test_da" in df_final.columns:
    df_gap["gap_da"]   = (df_final["train_da"]  - df_final["test_da"]).round(4)
if "train_pnl_ratio" in df_final.columns and "test_pnl_ratio" in df_final.columns:
    df_gap["gap_pnl"]  = (df_final["train_pnl_ratio"] - df_final["test_pnl_ratio"]).round(4)

display(df_gap)
print("  Note: positive gap_rmse = test worse; positive gap_da / gap_pnl = train better.")

# ===========================================================
# E — LSTM vs. Baselines
# ===========================================================
print(f"\n{'='*60}")
print("  E. LSTM vs. BASELINES")
print(f"{'='*60}")

_rows = []
for _, r in df_final.iterrows():
    _rows.append({
        "model":      r["selection_type"],
        "combo":      r["combo"],
        "rmse":       round(r["test_rmse"],  4) if "test_rmse"  in r.index else np.nan,
        "da":         round(r["test_da"],    4) if "test_da"    in r.index else np.nan,
        "pnl_ratio":  round(r["test_pnl_ratio"], 4) if "test_pnl_ratio" in r.index else np.nan,
    })
for _, b in df_baseline.iterrows():
    _rows.append({
        "model":     b["model"],
        "combo":     "—",
        "rmse":      round(b["rmse"], 4)      if pd.notna(b.get("rmse"))      else np.nan,
        "da":        round(b["da"],   4)       if pd.notna(b.get("da"))        else np.nan,
        "pnl_ratio": round(b["pnl_ratio"], 4) if pd.notna(b.get("pnl_ratio")) else np.nan,
    })

df_compare = pd.DataFrame(_rows)
display(df_compare)

# ===========================================================
# F — Training Loss Curves
# ===========================================================
print(f"\n{'='*60}")
print("  F. TRAINING LOSS CURVES (final retrain)")
print(f"{'='*60}")

_final_hist_files = sorted(_hist.glob("FINAL_*.csv"))
if _final_hist_files:
    fig, axes = plt.subplots(1, len(_final_hist_files),
                             figsize=(6 * len(_final_hist_files), 4), squeeze=False)
    for ax, hf in zip(axes[0], _final_hist_files):
        _tag = hf.stem.replace("FINAL_FINAL_", "").replace("FINAL_", "")
        _df_h = pd.read_csv(hf)
        ax.plot(_df_h["loss"], label="train loss", color="steelblue")
        if "val_loss" in _df_h.columns:
            ax.plot(_df_h["val_loss"], label="val loss", color="orange", linestyle="--")
        ax.set_title(_tag, fontsize=9)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        ax.legend(fontsize=8)
    fig.suptitle(f"Training History — {ticker} {TARGET_TYPE}", fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("  No final history files found.")

# ===========================================================
# G — Residual Analysis
# ===========================================================
print(f"\n{'='*60}")
print("  G. PREDICTION RESIDUAL ANALYSIS")
print(f"{'='*60}")

_json_files = sorted(_logs.glob("FINAL_*_summary.json"))
for jf in _json_files:
    with open(jf, "r") as _f:
        _s = json.load(_f)

    _tag   = _s.get("tag", jf.stem)
    _combo = _s.get("combo", "")
    _true  = np.array(_s.get("test_true", []))
    _pred  = np.array(_s.get("test_pred", []))

    if _true.size == 0 or _pred.size == 0:
        print(f"  {_tag}: no test_true/test_pred in JSON — skipping.")
        continue

    _res = _pred - _true          # residuals (positive = over-predicted)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

    # scatter: predicted vs actual
    ax1.scatter(_true, _pred, alpha=0.4, s=10, color="steelblue")
    _lo, _hi = min(_true.min(), _pred.min()), max(_true.max(), _pred.max())
    ax1.plot([_lo, _hi], [_lo, _hi], "r--", linewidth=1, label="perfect fit")
    ax1.set_xlabel("Actual Price")
    ax1.set_ylabel("Predicted Price")
    ax1.set_title(f"{_tag}\nPredicted vs Actual")
    ax1.legend(fontsize=8)

    # residuals over time
    ax2.plot(_res, color="darkorange", linewidth=0.8)
    ax2.axhline(0, color="k", linewidth=0.8, linestyle="--")
    ax2.set_xlabel("Test Day Index")
    ax2.set_ylabel("Residual (pred − actual)")
    ax2.set_title(f"{_combo}\nResiduals over Time")

    # residual histogram
    ax3.hist(_res, bins=30, color="slategray", edgecolor="white")
    ax3.axvline(0, color="red", linewidth=1, linestyle="--")
    ax3.set_xlabel("Residual")
    ax3.set_ylabel("Count")
    ax3.set_title("Residual Distribution")

    rmse_str = f"RMSE={_s.get('test_rmse', np.nan):.4f}"
    da_str   = f"DA={_s.get('test_da', np.nan):.4f}"
    fig.suptitle(f"{ticker} {TARGET_TYPE} — {_tag}   |   {rmse_str}   {da_str}", fontsize=10)
    plt.tight_layout()
    plt.show()

# ===========================================================
# H — Trade Log Analysis
# ===========================================================
print(f"\n{'='*60}")
print("  H. TRADE LOG ANALYSIS (test set)")
print(f"{'='*60}")

_pnl_files = sorted(_pnl_dir.glob("FINAL_*_TEST_pnl_log.csv"))
if not _pnl_files:
    print("  No TEST pnl log files found.")
else:
    _trade_rows = []
    for pf in _pnl_files:
        _tag = pf.stem.replace("FINAL_FINAL_", "").replace("FINAL_", "").replace("_TEST_pnl_log", "")
        try:
            _df_pnl = pd.read_csv(pf)
        except pd.errors.EmptyDataError:
            print(f"  ⚠️ Skipping unreadable/empty file: {pf.name}")
            continue
        _sells  = _df_pnl[_df_pnl["Action"] == "Sell"].copy()
        _total  = len(_sells)
        _wins   = (_sells["PnL"] > 0).sum()
        _losses = (_sells["PnL"] <= 0).sum()
        _wr     = round(_wins / _total, 4) if _total > 0 else 0.0
        _total_pnl = _sells["PnL"].sum()
        _avg_pnl   = _sells["PnL"].mean() if _total > 0 else 0.0
        _trade_rows.append({
            "model":             _tag,
            "total_trades":      _total,
            "wins":              _wins,
            "losses":            _losses,
            "win_ratio":         _wr,
            "total_pnl":         round(_total_pnl, 2),
            "avg_pnl_per_trade": round(_avg_pnl, 2),
        })

    if _trade_rows:
        df_trades = pd.DataFrame(_trade_rows)
        display(df_trades)
    else:
        print("  No valid trade log files to display.")

    # cumulative PnL plot — only over readable files
    _readable_pnl_files = []
    for pf in _pnl_files:
        try:
            pd.read_csv(pf)
            _readable_pnl_files.append(pf)
        except pd.errors.EmptyDataError:
            pass

    if _readable_pnl_files:
        fig, axes = plt.subplots(1, len(_readable_pnl_files),
                                 figsize=(6 * len(_readable_pnl_files), 4), squeeze=False)
        for ax, pf in zip(axes[0], _readable_pnl_files):
            _tag    = pf.stem.replace("FINAL_FINAL_", "").replace("FINAL_", "").replace("_TEST_pnl_log", "")
            _df_pnl = pd.read_csv(pf)
            _sells  = _df_pnl[_df_pnl["Action"] == "Sell"].copy().reset_index(drop=True)
            if "PnL" in _sells.columns and len(_sells) > 0:
                _cum = _sells["PnL"].cumsum()
                ax.plot(_cum, color="steelblue", linewidth=1.2)
                ax.axhline(0, color="k", linestyle="--", linewidth=0.8)
                ax.fill_between(range(len(_cum)), _cum, 0,
                                where=(_cum >= 0), alpha=0.2, color="green", label="profit")
                ax.fill_between(range(len(_cum)), _cum, 0,
                                where=(_cum < 0),  alpha=0.2, color="red",   label="loss")
                ax.set_xlabel("Trade #")
                ax.set_ylabel("Cumulative P&L")
                ax.legend(fontsize=8)
            else:
                ax.text(0.5, 0.5, "No trades", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(_tag, fontsize=9)

        fig.suptitle(f"Cumulative P&L — {ticker} {TARGET_TYPE} (Test Set)", fontsize=11)
        plt.tight_layout()
        plt.show()
    else:
        print("  No readable trade log files for plotting.")

print("\nAnalysis complete.")

In [ ]:
# ============================================================
# Indicator Frequency in Best-Performing Feature Combinations
# AAPL, AMZN, GOOGL — all 3 selected models per ticker (9 combos total)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

best_combos = {
    "AAPL": {
        "BEST_VAL_RMSE": "MACD+SMA+EMA+BB+MOM",
        "BEST_VAL_DA":   "SMA+EMA+BB+MOM",
        "BEST_VAL_PNL":  "SMA+EMA+BB+VOL",
    },
    "AMZN": {
        "BEST_VAL_RMSE": "MACD+EMA+BB+VOL",
        "BEST_VAL_DA":   "RSI+MACD+VOL",
        "BEST_VAL_PNL":  "MACD+SMA",
    },
    "GOOGL": {
        "BEST_VAL_RMSE": "MACD+SMA+EMA+BB+MOM",
        "BEST_VAL_DA":   "SMA+MOM",
        "BEST_VAL_PNL":  "RSI+MACD+EMA+MOM+VOL",
    },
}

all_indicators = ["RSI", "MACD", "SMA", "EMA", "BB", "MOM", "VOL"]
tickers        = ["AAPL", "AMZN", "GOOGL"]
selection_types = ["BEST_VAL_RMSE", "BEST_VAL_DA", "BEST_VAL_PNL"]

# ── Build presence matrix: indicator × ticker (count out of 3 models) ──
presence = {t: {ind: 0 for ind in all_indicators} for t in tickers}
for ticker, models in best_combos.items():
    for sel, combo in models.items():
        groups = combo.split("+")
        for ind in all_indicators:
            if ind in groups:
                presence[ticker][ind] += 1

# ── Figure 1: Grouped bar chart (indicator × ticker) ──────────────────
x     = np.arange(len(all_indicators))
width = 0.25
colors = {"AAPL": "#4C72B0", "AMZN": "#DD8452", "GOOGL": "#55A868"}

fig, ax = plt.subplots(figsize=(12, 5))
for i, ticker in enumerate(tickers):
    counts = [presence[ticker][ind] for ind in all_indicators]
    ax.bar(x + i * width, counts, width, label=ticker, color=colors[ticker], edgecolor="white")

ax.set_xticks(x + width)
ax.set_xticklabels(all_indicators, fontsize=12)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["0", "1", "2", "3 (all)"], fontsize=10)
ax.set_ylabel("Number of best models containing indicator\n(out of 3 per ticker)", fontsize=11)
ax.set_title("Indicator Frequency in Best-Performing Feature Combinations\n(AAPL · AMZN · GOOGL)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("indicator_frequency_grouped.png", dpi=150)
plt.show()

# ── Figure 2: Heatmap (indicator × ticker) ────────────────────────────
matrix = np.array([[presence[t][ind] for t in tickers] for ind in all_indicators])

fig2, ax2 = plt.subplots(figsize=(6, 6))
im = ax2.imshow(matrix, cmap="YlGn", vmin=0, vmax=3, aspect="auto")

ax2.set_xticks(range(len(tickers)))
ax2.set_xticklabels(tickers, fontsize=12)
ax2.set_yticks(range(len(all_indicators)))
ax2.set_yticklabels(all_indicators, fontsize=12)

for i, ind in enumerate(all_indicators):
    for j, ticker in enumerate(tickers):
        val = matrix[i, j]
        ax2.text(j, i, str(val), ha="center", va="center",
                 fontsize=14, fontweight="bold",
                 color="black" if val < 2.5 else "white")

cbar = plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
cbar.set_label("Appearances (max 3)", fontsize=10)
ax2.set_title("Indicator Presence Heatmap\n(Best Models per Ticker)", fontsize=12)
plt.tight_layout()
plt.savefig("indicator_frequency_heatmap.png", dpi=150)
plt.show()

# ── Print summary table ───────────────────────────────────────────────
print(f"\n{'Indicator':<10}", end="")
for t in tickers:
    print(f"  {t:<8}", end="")
print("  TOTAL")
print("─" * 45)
for ind in all_indicators:
    total = sum(presence[t][ind] for t in tickers)
    print(f"{ind:<10}", end="")
    for t in tickers:
        print(f"  {presence[t][ind]:<8}", end="")
    print(f"  {total}")

In [ ]:
# ============================================================
# Config visualization cell 
# Purpose:
# - avoid hard-coded Windows paths
# - load data/model using config.yaml and run_root-style folders
# - create the same test-set prediction figure in a reproducible way
# ============================================================

from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
import importlib

import indicators as _ind_mod
importlib.reload(_ind_mod)
from indicators import add_indicators

# ------------------------------------------------------------
# Visualization settings
# ------------------------------------------------------------
_TICKER = "GOOGL"
_COMBO_NAME = "MACD+SMA+EMA+BB+MOM"
_MODEL_TAG = f"FINAL_FINAL_BEST_VAL_RMSE_{_COMBO_NAME}"

# IMPORTANT:
# The paths below are derived from config.yaml / global setup.
# No local Windows paths are hard-coded here.
_RUN_DIR = runs_dir / f"{_TICKER}_{TARGET_TYPE}_experiment"
_MODEL_PATH = _RUN_DIR / "models" / f"{_MODEL_TAG}.keras"
_PLOT_DIR = _RUN_DIR / "plots"
_PLOT_DIR.mkdir(parents=True, exist_ok=True)

_WINDOW = int(window_size)
_TRAIN_FRAC = float(VAL_END)  # final retrain uses train + validation = first 80%

# Feature columns in the exact order used for this selected model
_FEAT_COLS = [
    "Close", "Open",
    "MACD", "MACD_signal", "MACD_hist",
    "SMA", "EMA",
    "BB_upper", "BB_lower", "BB_middle",
    "Momentum",
]

# ------------------------------------------------------------
# Load ticker data without hard-coded paths
# Priority:
# 1) config paths csv_folder / "{ticker}.csv"
# 2) any matching file beginning with ticker in csv_folder
# 3) project data_loader fallback
# ------------------------------------------------------------
_csv_path = csv_folder / f"{_TICKER}.csv"

if not _csv_path.exists():
    _matches = sorted(csv_folder.glob(f"{_TICKER}*.csv"))
    _csv_path = _matches[0] if _matches else None

if _csv_path is not None and _csv_path.exists():
    _df = pd.read_csv(_csv_path)

    if "Date" not in _df.columns:
        raise ValueError(f"'Date' column not found in {_csv_path}. Available columns: {list(_df.columns)}")

    _df["Date"] = pd.to_datetime(
        _df["Date"].astype(str).str.split(" ").str[0],
        errors="coerce"
    )
else:
    _df_list = load_stock_data(str(csv_folder), [_TICKER])
    if not _df_list:
        raise FileNotFoundError(f"No CSV or loader data found for ticker {_TICKER} in {csv_folder}")

    _df = _df_list[0].reset_index()
    if "Date" not in _df.columns:
        raise ValueError(f"'Date' column missing after load_stock_data() for ticker {_TICKER}")

    _df["Date"] = pd.to_datetime(_df["Date"], errors="coerce")

_df = _df.dropna(subset=["Date"]).set_index("Date").sort_index()

# ------------------------------------------------------------
# Rebuild features and target
# ------------------------------------------------------------
_df = add_indicators(_df, config.get("technical_indicators", {}))
_df["Target"] = _df["Close"].shift(-1)  # next-day close
_df = _df.dropna().copy()

_missing_features = [c for c in _FEAT_COLS if c not in _df.columns]
if _missing_features:
    raise ValueError(f"Missing selected feature columns for visualization: {_missing_features}")

_df_sel = _df[_FEAT_COLS + ["Target"]].copy()

# ------------------------------------------------------------
# Train/test split matching final retrain logic: first 80% / final 20%
# ------------------------------------------------------------
_n = len(_df_sel)
_i_split = max(1, min(int(_n * _TRAIN_FRAC), _n - 1))

_df_train = _df_sel.iloc[:_i_split].copy()
_df_test = _df_sel.iloc[_i_split:].copy()

# Fit scalers on non-test data only, matching the final retraining logic
_scaler_X = MinMaxScaler().fit(_df_train[_FEAT_COLS])
_scaler_y = MinMaxScaler().fit(_df_train[["Target"]])

_Xs = _scaler_X.transform(_df_test[_FEAT_COLS])
_ys = _scaler_y.transform(_df_test[["Target"]])

_df_ts = pd.DataFrame(_Xs, columns=_FEAT_COLS, index=_df_test.index)
_df_ts["Target"] = _ys

# ------------------------------------------------------------
# Build LSTM windows
# ------------------------------------------------------------
_X_seq, _y_seq, _seq_dates = [], [], []

for _i in range(_WINDOW, len(_df_ts)):
    _X_seq.append(_df_ts[_FEAT_COLS].iloc[_i - _WINDOW:_i].values)
    _y_seq.append(_df_ts["Target"].iloc[_i])
    _seq_dates.append(_df_ts.index[_i])

_X_seq = np.asarray(_X_seq, dtype=np.float32)
_y_seq = np.asarray(_y_seq, dtype=np.float32)
_seq_dates = pd.DatetimeIndex(_seq_dates)

if len(_X_seq) == 0:
    raise ValueError("Not enough test data to create LSTM sequences.")

# ------------------------------------------------------------
# Load final model and predict
# ------------------------------------------------------------
if not _MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found: {_MODEL_PATH}\n"
        "Check _MODEL_TAG, _COMBO_NAME, and the run folder generated by the final training step."
    )

_model = keras.models.load_model(_MODEL_PATH)
_y_pred_s = _model.predict(_X_seq, verbose=0)

_y_pred = _scaler_y.inverse_transform(_y_pred_s).flatten()
_y_true = _scaler_y.inverse_transform(_y_seq.reshape(-1, 1)).flatten()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
_rmse = np.sqrt(mean_squared_error(_y_true, _y_pred))
_mae = mean_absolute_error(_y_true, _y_pred)
_r2 = r2_score(_y_true, _y_pred)
_mask = _y_true != 0
_mape = np.mean(np.abs((_y_true[_mask] - _y_pred[_mask]) / _y_true[_mask])) * 100 if _mask.any() else np.nan

print(f"Ticker={_TICKER}")
print(f"Run folder={_RUN_DIR}")
print(f"Model path={_MODEL_PATH}")
print(f"RMSE={_rmse:.3f}  MAE={_mae:.3f}  MAPE={_mape:.2f}%  R²={_r2:.4f}")

# ------------------------------------------------------------
# Plot: main view + zoomed inset
# ------------------------------------------------------------
_ZOOM_N = 60

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(_seq_dates, _y_true, label="Actual Price", linewidth=1.8, zorder=3)
ax.plot(_seq_dates, _y_pred, label="Predicted Price", linewidth=1.5, linestyle="--", zorder=3)

_z_end = min(_ZOOM_N, len(_seq_dates) - 1)
ax.axvspan(_seq_dates[0], _seq_dates[_z_end], alpha=0.08, zorder=1)

ax.set_title(
    f"{_TICKER} — Actual vs Predicted Close Price · Test Set\n"
    f"Feature combo: {_COMBO_NAME}  |  "
    f"RMSE = {_rmse:.2f}  ·  MAPE = {_mape:.2f}%  ·  R² = {_r2:.3f}",
    fontsize=12,
    pad=10
)
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Price (USD)", fontsize=11)
ax.legend(fontsize=11, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.grid(alpha=0.25)

_zd = _seq_dates[:_z_end + 1]
_zt = _y_true[:_z_end + 1]
_zp = _y_pred[:_z_end + 1]

axins = ax.inset_axes([0.58, 0.06, 0.40, 0.42])
axins.plot(_zd, _zt, linewidth=1.5)
axins.plot(_zd, _zp, linewidth=1.2, linestyle="--")
axins.set_xlim(_zd[0], _zd[-1])
axins.set_ylim(
    min(_zt.min(), _zp.min()) * 0.992,
    max(_zt.max(), _zp.max()) * 1.008
)
axins.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
axins.xaxis.set_major_locator(mdates.WeekdayLocator(interval=3))
plt.setp(axins.get_xticklabels(), rotation=30, ha="right", fontsize=7)
axins.tick_params(axis="y", labelsize=7)
axins.set_title(f"Zoomed: first {_ZOOM_N} test days", fontsize=8)
axins.grid(alpha=0.3)

ax.indicate_inset_zoom(axins, alpha=0.7)

plt.tight_layout()
_save = _PLOT_DIR / f"{_TICKER}_test_predictions_zoomed_config_based.png"
plt.savefig(_save, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved → {_save}")


In [ ]:
# ============================================================
# GOOGL — Directional Correctness & Buy/Hold/Sell Signal Comparison
# Requires: _y_true, _y_pred, _seq_dates, _df, _RUN_DIR
# from the previous prediction cell
# ============================================================
from sklearn.metrics import confusion_matrix
import matplotlib.patches as mpatches

_THRESHOLD = 0.002   # 0.2 % — same as trading.threshold_pct in config

# ── Current-day close at each prediction date ─────────────────────────
# _seq_dates[k] is the target date (= next-day date in the raw data).
# The "current" close is the Close at that same date index in _df_sel.
_current_close = np.array([
    _df_sel["Close"].iloc[_i_split + _WINDOW + k]
    for k in range(len(_seq_dates))
], dtype=float)

# ── Derive signals: Buy=1, Hold=0, Sell=-1 ────────────────────────────
def _to_signal(pred_px, curr_px, thr):
    delta = (pred_px - curr_px) / curr_px
    return np.where(delta > thr, 1, np.where(delta < -thr, -1, 0))

_sig_pred   = _to_signal(_y_pred,  _current_close, _THRESHOLD)
_sig_actual = _to_signal(_y_true,  _current_close, _THRESHOLD)

_correct    = (_sig_pred == _sig_actual)
_n          = len(_sig_pred)
_da         = _correct.mean() * 100

print(f"Directional Accuracy  : {_da:.1f}%  ({_correct.sum()} / {_n})")
print(f"Pred   — Buy:{(_sig_pred==1).sum()}  Hold:{(_sig_pred==0).sum()}  Sell:{(_sig_pred==-1).sum()}")
print(f"Actual — Buy:{(_sig_actual==1).sum()}  Hold:{(_sig_actual==0).sum()}  Sell:{(_sig_actual==-1).sum()}")

# ── Figure: 3 stacked panels ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          gridspec_kw={"height_ratios": [3, 1.8, 1.8]})
fig.suptitle(
    f"GOOGL — Directional Correctness & Signal Comparison · Test Set\n"
    f"Combo: {_COMBO_NAME}  |  Threshold: {_THRESHOLD*100:.1f}%  |  "
    f"Directional Accuracy: {_da:.1f}%",
    fontsize=13, y=1.01
)

# ── Panel 1: Price with correct/wrong markers ─────────────────────────
ax0 = axes[0]
ax0.plot(_seq_dates, _y_true, color="#1565C0", linewidth=1.6,
         label="Actual Price", zorder=2)
ax0.plot(_seq_dates, _y_pred, color="#757575", linewidth=1.0,
         linestyle="--", alpha=0.7, label="Predicted Price", zorder=2)

_c_ok  = _seq_dates[_correct]
_c_bad = _seq_dates[~_correct]
ax0.scatter(_c_ok,  _y_true[_correct],  color="#2E7D32", s=18,
            zorder=4, label="Correct direction")
ax0.scatter(_c_bad, _y_true[~_correct], color="#C62828", s=18,
            marker="x", zorder=4, label="Wrong direction")

ax0.set_ylabel("Price (USD)", fontsize=11)
ax0.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax0.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax0.get_xticklabels(), rotation=30, ha="right")
ax0.legend(fontsize=10, loc="upper left")
ax0.grid(alpha=0.25)

# ── Panel 2: Signal timeline side by side ─────────────────────────────
ax1 = axes[1]
_COLORS = {1: "#2E7D32", 0: "#F9A825", -1: "#C62828"}
_LABELS = {1: "Buy", 0: "Hold", -1: "Sell"}
_OFFSET = 0.15   # vertical jitter to separate actual vs pred

for _sig, _label, _y_offset, _marker in [
    (_sig_actual, "Actual", +_OFFSET, "o"),
    (_sig_pred,   "Pred",   -_OFFSET, "^"),
]:
    for _sv in [1, 0, -1]:
        _mask_s = (_sig == _sv)
        if _mask_s.any():
            ax1.scatter(
                _seq_dates[_mask_s],
                np.full(_mask_s.sum(), _sv + _y_offset),
                color=_COLORS[_sv], marker=_marker,
                s=22, alpha=0.85,
                label=f"{_label} — {_LABELS[_sv]}" if _label == "Actual" else None,
                zorder=3,
            )

ax1.set_yticks([-1, 0, 1])
ax1.set_yticklabels(["Sell", "Hold", "Buy"], fontsize=11)
ax1.set_ylabel("Signal", fontsize=11)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax1.get_xticklabels(), rotation=30, ha="right")
ax1.grid(axis="x", alpha=0.25)
ax1.axhline(0, color="gray", linewidth=0.5, linestyle=":")

_actual_patch = mpatches.Patch(facecolor="none", edgecolor="black",
                                label="● Actual  ▲ Predicted")
_buy_p  = mpatches.Patch(color=_COLORS[1],  label="Buy")
_hold_p = mpatches.Patch(color=_COLORS[0],  label="Hold")
_sell_p = mpatches.Patch(color=_COLORS[-1], label="Sell")
ax1.legend(handles=[_actual_patch, _buy_p, _hold_p, _sell_p],
           fontsize=9, loc="upper right", ncol=2)

# ── Panel 3: Confusion matrix heatmap ─────────────────────────────────
ax2 = axes[2]
_LABELS_ORD = [-1, 0, 1]
_TICK_NAMES  = ["Sell", "Hold", "Buy"]
_cm = confusion_matrix(_sig_actual, _sig_pred, labels=_LABELS_ORD)

_im = ax2.imshow(_cm, cmap="Blues", aspect="auto")
ax2.set_xticks(range(3)); ax2.set_xticklabels(_TICK_NAMES, fontsize=11)
ax2.set_yticks(range(3)); ax2.set_yticklabels(_TICK_NAMES, fontsize=11)
ax2.set_xlabel("Predicted Signal", fontsize=11)
ax2.set_ylabel("Actual Signal", fontsize=11)
ax2.set_title("Signal Confusion Matrix", fontsize=11)

for _ri in range(3):
    for _ci in range(3):
        _val = _cm[_ri, _ci]
        ax2.text(_ci, _ri, str(_val), ha="center", va="center",
                 fontsize=13, fontweight="bold",
                 color="white" if _val > _cm.max() * 0.6 else "black")

plt.colorbar(_im, ax=ax2, fraction=0.035, pad=0.04, label="Count")

fig.tight_layout()
_save2 = os.path.join(_RUN_DIR, "plots", "GOOGL_signal_comparison.png")
plt.savefig(_save2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {_save2}")